# 리포트 16 — 앙각 커버리지 — 어느 각도까지 유효한가

> 관측 앙각을 0° 에서 −90° 까지 내리며 같은 표적을 재면, 커버리지를 정하는 것은 표적이 아니라 **우리가 고른 분석 대역과 잣대**다. 대역을 날개끝 주파수를 따라 옮기면 −75° 에서도 프로펠러 대역이 살아 있고, 8/11 덱의 대역을 고정한 채 내려가면 같은 자리가 바닥에 앉는다. 잣대를 고를 때마다 **광선 예산이 함께 움직인다**는 것이 이 권의 두 번째 결론이다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현** 을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 | 만든 곳 |
|---|---|---|
| 1 | 앙각 7 점을 10 m 한 자리에서 재고, 47 행 중 46 행만 판정에 쓴다 | `_parts/78_el-sweep-design.ipynb` |
| 2 ⭐ | −75° 에서 추적 대역 몫은 고정 대역보다 38.72 dB 크고, 그 차이를 만든 것은 대역을 어디에 놓았는가 하나다 | `_parts/79_el-band-tracking.ipynb` |
| 3 | 물리 상한 위 누설은 우리 팔 0.26~15.65 %, 스톡 PathSolver 두 예산 5.12~87.02 % 이고, 그 위 끝은 평평한 스펙트럼이 받는 점수다 | `_parts/80_el-above-tip-limit.ipynb` |
| 4 | 나딧 잔여 −38.31 dB 의 64 % 는 광선 격자 표본화 잡음이고, 5° 만 기울면 −11.88 dB 로 열린다 | `_parts/82_el-nadir-floor.ipynb` |
| 5 | el 0 에서 광선을 360 배 늘리면 정지 성분은 0.03 dB 안에 모이고, 같은 한 계단이 el −75 의 레벨을 12.55 dB 옮긴다 | `_parts/87_budget-not-physics.ipynb` |

⭐ 표시한 절 하나만 읽어도 이 권의 결론은 선다.

숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 절 끝 «출처» 표가 그 파일과 키다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.

전체 목차는 [reports/README.md](README.md) 이고, 열일곱 권의 지도는 [리포트 1 «이 연구가 묻는 것과 답한 방식»](01_map.ipynb) 다.


---

## 절 1. 앙각 7 점을 10 m 한 자리에서 재고, 47 행 중 46 행만 판정에 쓴다



> ### 한 일
> **관측 앙각 7 점을 10 m 구면 한 자리에 고정해 세 팔로 같은 표적을 재고, 완결된 46 행만 인용 대상으로 갈랐다.**

### 결과
1. 잰 자리는 하나다 — 반경 10 m [^1] 구면, 방위 0° [^2], 앙각 +0° [^3] 에서 -90° [^4] 까지 7 점.
2. 한 앙각마다 자세 4,096 개 [^5] 를 PRF 19,700 Hz [^6] 로 태웠고, 표적은 matrice4e [^7] 하나다.
3. 그 자리는 원거리장 경계의 안쪽이라 근접장 판이고, 우리 커널의 조명 규약은 «spherical wave at 10 m [^8]» 다.
4. 원장 47 행 중 46 행이 `n_missing = 0` 이고, 물리 스위치 팔의 −15°(0 개 [^9] 빠짐) 와 −45°(0 개 [^10] 빠짐) 두 행은 판정에서 뺀다.
5. 경로 수는 팔 사이에서 예산 축이고 한 팔 안에서 기하 축이다 — 같은 7 점에서 규칙값 팔은 6 [^11]~13 [^12] 개, `--spp` 로 광선을 22.5 배 올린 팔은 127 [^13]~352 [^14] 개를 센다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 기하 | 반경 10 m 구면 위에서 방위 하나·앙각 일곱. 송신과 수신은 같은 자리다(baseline 0) — `benchmark/elevation_sweep_md.py:83,175` |
| 표적·회전 | matrice4e 메쉬 전체(동체·팔·로터·짐벌)에 첫 충돌 가림. 로터 넷은 덱과 같은 결정론 RPM 이라 바뀌는 축은 앙각 하나다 |
| 조명 | 우리 커널은 구면파로 조명하고, PathSolver 는 송수신 위치를 실제 기하로 놓는다. 광선 수의 기본값을 거리만 보는 규칙 `(R/3)²×1M` 이 정하고, p250M·p1G·p4G 팔은 `--spp` 로 그 값을 덮어쓴다 |
| 분석 대역 | 추적 대역은 앙각마다 그 앙각의 날개끝 주파수로 다시 잡고, 고정 대역은 덱의 −15° 대역을 그대로 쓴다 — 두 정의는 «대역은 두 가지로 잰다» 절에 원장 문장 그대로 싣는다 |
| 완결성 | `rows[i].n_missing` 은 시계열에 0 으로 남은 자세 수다. 이 조각의 표는 `(engine, el_deg, n_missing == 0)` 으로 행을 찾아 만들었다 |

### 재현

```bash
SIONNA2_GPU=3 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/elevation_sweep_md.py --engine ours --shard 0 --nshards 8
SIONNA2_GPU=3 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/elevation_sweep_md.py --engine sionna --shard 0 --nshards 8
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/elevation_sweep_md.py --merge
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_fig_el_geometry.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/render_el15_scene.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/build_el15_scenario_fig.py
```

| | |
|---|---|
| 출력 | `outputs/elevation_sweep_md.json`, `outputs/elevation_sweep_md.npz`, `outputs/figures/ch1_f0_geometry.png`, `outputs/figures/el15_scenario.png` |
| 소요 | 한 행마다 GPU 누적 14 분 ~ 2 시간 21 분 · 샤드 8 개 병렬. 병합과 그림은 CPU 로 수 초 |
| 비고 | `--merge` 는 샤드 폴더 `outputs/elev_sweep_shards/` 를 읽어 원장 두 개를 다시 쓴다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [리포트 9 절 1 «자세와 가림»](09_microdoppler-limits.ipynb) | 지상 레이더가 기체를 아래에서 본다는 **기체 자세** 축 — 이 권이 바꾸는 앙각은 **수신 기하**이고 그 자세와 다른 축이다 |
| [리포트 8 절 3 «두 엔진»](08_3_pattern.ipynb) | 두 엔진이 날개끝 주파수 아래에서 겹치고 그 위에서 갈린다는 −15° 한 점의 결과 |

---


## 잰 자리는 한 자리다

앙각 하나만 바꾼다. 거리·방위·로터 RPM·자세 격자는 같은 값으로 얼려 뒀으므로, 팔 사이와 앙각 사이에서 달라진 것은 시선 방향 하나다.

로터 설정을 원장은 이렇게 적는다 — «덱과 같은 결정론 패턴(OU 프리셋 아님) — 축을 하나만 바꾼다 [^15]».


| 설정 | 값 |
|---|---|
| 표적 | matrice4e [^7] |
| 반송파 | 3.50e+09 Hz [^16] |
| 거리 — 구면 반경 | 10 m [^1] |
| 방위 | 0° [^2] |
| 앙각 | +0° [^3] 에서 -90° [^4] 까지 7 점 |
| 자세 | 4,096 개 [^5] |
| PRF | 19,700 Hz [^6] |
| 플래시 박자 — 예측 입력 | 126.67 Hz [^17] |
| 날개끝 주파수 — 앙각 0° | 1272.9 Hz [^18] |


## 그 자리는 어느 원거리장 정의를 쓰느냐로 갈린다

> ⭐사용자 지시로 10 m 고정. ⚠원거리장 경계 2D²/λ ≈ 14.08 m 의 **안쪽**이라 근거리장 판이다 — 우리 커널은 range_m 구면파로 처리하고 PathSolver 는 실제 기하라 둘 다 다룰 수 있지만, 평면파 원거리장 값과 직접 비교하면 안 된다. [^19]

matrice4e·3.5 GHz 에서 2D²/λ 는 **D 를 무엇으로 잡느냐에 따라 두 값**이다 — 수평 최대치수 0.59 m [^20] 로 잡으면 8.26 m [^21], 메쉬 3D 대각 0.78 m [^22] 로 잡으면 14.08 m [^23] 다. 10 m 는 그 **사이**에 있다. 곧 이 판은 보수적 정의(3D 대각)에서 경계 안쪽이고, 수평 최대치수 정의에서는 경계 밖이다. 이 권은 «경계» 를 쓸 때 **정의를 값과 함께** 적는다.


그것이 이 판에서 실제로 바꾸는 것은 **둘**이다.

1. 우리 커널은 표적을 구면파로 조명한다 — 원장의 조명 규약은 «spherical wave at 10 m [^8]» 다.
2. 평면파 원거리장에서 잰 레벨과 이 판의 레벨을 같은 줄에 놓으려면 **환산 몫을 적는다.** 그 몫의 크기는 이미 재어 뒀다 — 같은 기체·같은 반송파·같은 얼린 격자·앙각 −15° 한 점에서 구면파와 평면파의 차이는 8 m 에서 레벨 0.52 dB [^24] · 맵 코사인 0.998 [^25], 경계 밖 15 m 에서 0.29 dB [^26] · 0.997 [^27] 다. 차이는 거리와 함께 매끄럽게 줄고 **경계를 그대로 지나간다.**


다음 셋은 경계와 무관한 **판 조건**이다 — 어느 거리에서도 같다.

- PathSolver 는 송수신 위치를 실제 기하로 놓는다. 그 계산이 받는 것은 두 자리의 좌표뿐이다.
- 광선 수 11,111,111 개 [^28] 는 `(R/3)²×1M` 이라는 **거리만 보는** 규칙이 정한다.
- 표면 격자는 «얼린 격자(자세 합집합 bbox), λ/12 [^29]» 다 — 자세마다 다시 잡지 않으므로 앙각 사이의 차이는 격자가 아니라 자세와 시선에서 온다.


![experiment scenario rendered with Sionna RT](../outputs/figures/el15_scenario.png)

**그림 1.** 이 판은 표적을 어느 자리에서 보고, 그 자리에서 무엇이 보이나?

위 칸은 기하다 — 15 m 구면 위 앙각 일곱 점과, 그 안쪽을 지나는 원거리장 경계선. 아래 칸은 그 일곱 자리에서 **레이더가 실제로 보는 표적**이고 `benchmark/render_el15_scene.py` 가 Sionna RT 로 낸 렌더다. 왼쪽 끝(앙각 0°)은 로터를 옆에서 봐 블레이드가 선으로 보이고, 오른쪽 끝(−90°)은 로터 원반이 열리는 대신 **동체가 가운데를 덮는다** — 아래 절이 가르는 두 몫이 이것이다.


## 메쉬를 통째로 넣은 대가는 두 몫이 겹쳐 있다

메쉬는 통째로 넣었다 — 동체·팔·로터·짐벌이 다 들어 있고 첫 충돌 가림이 켜져 있다. 그래서 앙각을 내리면 두 가지가 함께 움직인다.

1. 날개끝 속도의 시선 방향 성분이 cos(el) 로 준다.
2. 동체가 로터와 센서 사이로 들어온다 — 나딧에서는 동체 원반이 로터를 덮는다.


아래 표의 «날개끝 주파수» 는 1 번만 담는 **입력값**이다. `f_tip_at()`(`benchmark/elevation_sweep_md.py:205`) 이 로터 지름·회전수에서 f_tip = 2·(2π f_rev R)/λ · cos(el) 로 내며, 앙각 0° 의 1272.9 Hz [^18] 에서 −90° 의 0.0 Hz [^30] 로 간다. 그것은 cos(el) 열에 앙각 0° 값을 곱한 수와 같고, 원장에서 앙각 0° 를 가진 여섯 팔이 전부 같은 값을 싣는다 — 이 열은 앙각 하나의 함수다. 이 표에서 이 판이 잰 열은 `빠진 자세` 하나이고, 일곱 행 모두 0 이라 우리 커널의 일곱 점을 그대로 인용한다.


| 앙각 [°] | cos(el) | 날개끝 주파수 [Hz] | 빠진 자세 |
|---|---|---|---|
| +0 | 1.0000 | 1272.9 | 0 |
| -15 | 0.9659 | 1229.5 | 0 |
| -30 | 0.8660 | 1102.4 | 0 |
| -45 | 0.7071 | 900.1 | 0 |
| -60 | 0.5000 | 636.5 | 0 |
| -75 | 0.2588 | 329.5 | 0 |
| -90 | 0.0000 | 0.0 | 0 |

출처 [^31]


2 번 몫은 대조군이 가른다. 동체의 «면만» 빼고 정점을 남겨 bbox 와 광선 격자를 보존하는 `ours_free`(`benchmark/elevation_sweep_md.py:117-122, 132-133`) 가 그것이고, 축은 «동체가 막느냐» 하나다. 그 팔은 스크립트에 배선돼 있고 원장에도 샤드에도 행이 없어 «다음 단계» 첫 줄이다.

지금 배선은 `keep = np.asarray(fp.g) == "prop"` 이라 prop 아닌 면을 **전부** 뺀다 — `DRONE_GROUP_MAT` 기준으로 body·canopy·arm·motor·gear·camera·accent·battery·pcb 가 함께 빠지므로 가림과 정적 산란체(DC 분모)가 한 축에 묶인다. 가림만 가르려면 `keep` 을 «body·canopy 만 뺀다» 로 좁혀 돌린다.


## 세 팔이 같은 자리에서 낸 것

![micro-Doppler maps versus elevation](../outputs/figures/ch1_f1_maps.png)

**그림 2.** 세 팔은 같은 자리에서 앙각을 내릴 때 무엇을 냈나?

위 줄이 우리 커널(SBR + PO), 가운데가 PathSolver, 아래가 광선 예산을 올린 PathSolver(`sionna_p250000000`) 다. 판마다 자기 최댓값으로 정규화했으므로 판 사이 레벨 비교는 이 그림 밖이다. 그림은 7 점 중 네 점을 싣고, 일곱 점 전부의 완결성은 아래 표에 있다.


## 대역은 두 가지로 잰다

- 추적 대역 — ⭐정본 — 앙각마다 그 앙각의 f_tip 으로 0.35~1.0 배 [^32]
- 고정 대역 — 덱의 −15° 대역(430~1229 Hz) 고정 — 앙각이 내려가면 비어 간다 [^33]

둘을 함께 내는 이유는 «고정 대역을 쓰면 어디서 무너지나» 가 그 자체로 결과이기 때문이다. 그 판정은 **절 2** «대역 추적» 에 있다.


## 어느 행을 인용해도 되나 — 47 행 중 46 행

원장은 47 행이고 그중 46 행이 `n_missing = 0` 이다. 나머지 두 행은 물리 스위치 팔의 −15° 와 −45° 이고 빠진 자세가 각각 0 개 [^9] · 0 개 [^10] 다 — 그 자리에 0 이 박혀 있어 스펙트럼과 레벨이 그만큼 눌린다. 이 권은 그 두 행을 판정에서 뺀다.

아래 표가 세 팔 밖의 행 전부다. 예산 사다리(광선 1 G · 4 G)와 물리 스위치 팔이 여기 산다.


| 팔 | 앙각 [°] | 빠진 자세 | 경로 수 중앙값 |
|---|---|---|---|
| sionna_p1000000000 | +0 | 0 | 471 |
| sionna_p1000000000 | -15 | 0 | 736 |
| sionna_p1000000000 | -30 | 0 | 809 |
| sionna_p1000000000 | -45 | 0 | 1065 |
| sionna_p1000000000 | -60 | 0 | 1143 |
| sionna_p1000000000 | -75 | 0 | 1241 |
| sionna_p1000000000 | -90 | 0 | 1370 |
| sionna_p250000000_phys | +0 | 0 | 33 |
| sionna_p250000000_phys | -15 | 0 | 48 |
| sionna_p250000000_phys | -30 | 0 | 52 |
| sionna_p250000000_phys | -45 | 0 | 64 |
| sionna_p250000000_phys | -60 | 0 | 66 |
| sionna_p250000000_phys | -75 | 0 | 57 |
| sionna_p250000000_phys | -90 | 0 | 64 |
| sionna_p4000000000 | +0 | 0 | 2008 |
| sionna_p4000000000 | -15 | 0 | 2767 |
| sionna_p4000000000 | -45 | 0 | 4136 |
| sionna_p4000000000 | -60 | 0 | 4707 |
| sionna_p4000000000 | -75 | 2,048 | 5197 |
| sionna_phys | +0 | 0 | 5 |
| sionna_phys | -15 | 0 | 2 |
| sionna_phys | -30 | 0 | 7 |
| sionna_phys | -45 | 0 | 8 |
| sionna_phys | -60 | 0 | 4 |
| sionna_phys | -75 | 0 | 3 |
| sionna_phys | -90 | 0 | 6 |

출처 [^31]


이 권이 이 원장을 읽는 규칙은 셋이다.

1. 판정은 대역 몫으로 하고 `level_db` 는 **절 5** «광선 예산» 이 같은 엔진 안에서 다룬다 — 팔마다 정규화가 다르다.
2. −90° 에서 추적 대역의 폭이 0 이라 그 칸이 `null` 이다. `null` 은 0 이 아니라 «잴 수 없다» 는 표시이고, **절 2** «대역 추적» 이 «측정 불가» 로 적는다.
3. 행 번호는 병합할 때마다 밀린다. 이 조각의 표는 `(engine, el_deg, n_missing == 0)` 으로 행을 찾아 만들었다.


## 경로 수는 팔 사이에서만 예산 축이다

10 m 한 자리에서 규칙 `(R/3)²×1M` 은 11,111,111 개 [^28] 로 고정이고, 일곱 점의 예산이 모두 같은 값 하나다. 그래서 규칙값 팔의 6 [^11]~13 [^12] 개와 250M 팔의 127 [^13]~352 [^14] 개를 가른 것은 `--spp` 로 규칙값의 22.5 배를 쏜 설정 하나다(`benchmark/elevation_sweep_md.py:88,150`).

예산이 고정된 한 팔 **안에서** 경로 수가 앙각을 따라 움직이는 몫은 시선 기하가 정한다. 250M 팔은 앙각 0° 에서 −90° 로 가는 동안 단조로 늘고, 규칙값 팔은 6~13 사이에서 흔들려 그 추이를 내지 않는다 — 아래 두 표가 같은 열을 앙각별로 싣는다.

⇒ 팔 사이에서 경로 수가 다르면 예산 설정을 먼저 보고, 한 팔 안에서 달라지면 시선 기하를 본다. 인용한 네 수는 자세 4,096 개 중앙값의 최소·최대이고, 자세 하나하나의 경로 수는 그보다 넓게 흩어진다. 경로 수를 산란 세기로 읽는 해석은 [리포트 2 절 2 «경로가 무엇을 세나»](02_stock-engine.ipynb) 와 [리포트 17 절 1 «물리 스위치»](17_engine-physics.ipynb) 가 다룬다.


**PathSolver(`sionna`)**

| 앙각 [°] | 빠진 자세 | 경로 수 중앙값 |
|---|---|---|
| +0 | 0 | 9 |
| -15 | 0 | 7 |
| -30 | 0 | 6 |
| -45 | 0 | 12 |
| -60 | 0 | 13 |
| -75 | 0 | 13 |
| -90 | 0 | 12 |

출처 [^31]


**PathSolver(`sionna_p250000000`) — 광선 예산을 올린 팔**

| 앙각 [°] | 빠진 자세 | 경로 수 중앙값 |
|---|---|---|
| +0 | 0 | 127 |
| -15 | 0 | 159 |
| -30 | 0 | 211 |
| -45 | 0 | 268 |
| -60 | 0 | 287 |
| -75 | 0 | 323 |
| -90 | 0 | 352 |

출처 [^31]


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 가림만 끄고 산란체는 남기는 팔을 배선해 같은 일곱 점에서 돌린다 — 지금의 `ours_free` 는 프로펠러만 남기는 팔이라 분모까지 바뀐다 | 가림이 대역 몫을 얼마나 지우는지가 정적 성분 변화와 분리돼 나온다 | `benchmark/elevation_sweep_md.py` 가림 축 · **새 계산이 필요하다** |
| −60° 와 −75° 사이를 다섯 점 더 잰다 | 고정 대역이 대역외 바닥에 닿는 앙각이 15° 격자 안에서 특정된다 | `benchmark/elevation_sweep_md.py --els` · **절 2** «대역 추적» |
| 디스크에 8/8 로 차 있는 물리 팔 샤드를 병합한다 | 완결 행이 46 행에서 늘고 물리 팔이 두 점 아닌 다섯 점 이상에서 선다 | `benchmark/elevation_sweep_md.py --merge` (CPU) — 기존 원장을 다시 쓴다 |
| 평면파 조명으로 −90° 한 판을 더 돌린다 | 나딧 잔여에서 근접장 몫과 격자 몫이 직접 갈린다 | `benchmark/elevation_sweep_md.py` 조명 축 · **절 4** «나딧 잔여» |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 33개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/elevation_sweep_md.json` | `_meta.range_m` | 10 |
| [^2] | `outputs/report07_three_engines.json` | `_meta.az_deg` | 0 |
| [^3] | `outputs/elevation_sweep_md.json` | `_meta.elevations_deg[0]` | 0 |
| [^4] | `outputs/elevation_sweep_md.json` | `_meta.elevations_deg[6]` | -90 |
| [^5] | `outputs/elevation_sweep_md.json` | `rows[0].n_poses` | 4096 |
| [^6] | `outputs/elevation_sweep_md.json` | `_meta.prf_hz` | 19700 |
| [^7] | `outputs/elevation_sweep_md.json` | `_meta.drone` | matrice4e |
| [^8] | `outputs/elevation_sweep_md.json` | `_meta.ours_illumination` | spherical wave at 10 m |
| [^9] | `outputs/elevation_sweep_md.json` | `rows[41].n_missing` | 0 |
| [^10] | `outputs/elevation_sweep_md.json` | `rows[43].n_missing` | 0 |
| [^11] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_npaths_min_max[0]` | 6 |
| [^12] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_npaths_min_max[1]` | 13 |
| [^13] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_p250000000_npaths_min_max[0]` | 127 |
| [^14] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_p250000000_npaths_min_max[1]` | 352 |
| [^15] | `outputs/elevation_sweep_md.json` | `_meta.rotor_ko` | 덱과 같은 결정론 패턴(OU 프리셋 아님) — 축을 하나만 바꾼다 |
| [^16] | `outputs/elevation_sweep_md.json` | `_meta.fc_hz` | 3.5e+09 |
| [^17] | `outputs/elevation_sweep_md.json` | `_meta.f_flash_hz` | 126.7 |
| [^18] | `outputs/elevation_sweep_md.json` | `rows[0].f_tip_hz` | 1273 |
| [^19] | `outputs/elevation_sweep_md.json` | `_meta.range_why_ko` | ⭐사용자 지시로 10 m 고정. ⚠원거리장 경계 2D²/λ ≈ 14.08 m 의 **안쪽**이라 근… |
| [^20] | `outputs/report15_probe.json` | `airframes.matrice4e.physics.D_horizontal_m` | 0.5947 |
| [^21] | `outputs/report15_probe.json` | `airframes.matrice4e.physics.farfield_m` | 8.259 |
| [^22] | `outputs/report15_probe.json` | `airframes.matrice4e.physics.D_diag3d_m` | 0.7764 |
| [^23] | `outputs/report15_probe.json` | `airframes.matrice4e.physics.farfield_diag3d_m` | 14.08 |
| [^24] | `outputs/nearfield_sphere_vs_plane.json` | `ranges.8.level_diff_db` | 0.5152 |
| [^25] | `outputs/nearfield_sphere_vs_plane.json` | `ranges.8.map_cosine` | 0.9976 |
| [^26] | `outputs/nearfield_sphere_vs_plane.json` | `ranges.15.level_diff_db` | 0.2936 |
| [^27] | `outputs/nearfield_sphere_vs_plane.json` | `ranges.15.map_cosine` | 0.9974 |
| [^28] | `outputs/elevation_sweep_md.json` | `_meta.sionna_spp` | 11111111 |
| [^29] | `outputs/elevation_sweep_md.json` | `_meta.grid_ko` | 얼린 격자(자세 합집합 bbox), λ/12 |
| [^30] | `outputs/elevation_sweep_md.json` | `rows[6].f_tip_hz` | 0 |
| [^31] | `outputs/elevation_sweep_md.json` | `rows` | (47행 표) |
| [^32] | `outputs/elevation_sweep_md.json` | `_meta.band_track_ko` | ⭐정본 — 앙각마다 그 앙각의 f_tip 으로 0.35~1.0 배 |
| [^33] | `outputs/elevation_sweep_md.json` | `_meta.band_fixed_ko` | 덱의 −15° 대역(430~1229 Hz) 고정 — 앙각이 내려가면 비어 간다 |


---

## 절 2. −75° 에서 추적 대역 몫은 고정 대역보다 38.72 dB 크고, 그 차이를 만든 것은 대역을 어디에 놓았는가 하나다



> ### 한 일
> **앙각 7 점의 같은 시계열에서 프로펠러 대역을 두 가지로 잡아 서로 견주고, 같은 양을 세 정규화(전체 전력·반송파·대역외 기준띠)로 함께 냈다.**

### 결과
1. −75° 에서 추적 대역 몫은 -10.88 dB [^34], 고정 대역 몫은 -49.60 dB [^35] — 차이 +38.72 dB [^36] 다. 두 몫의 분모가 같은 시계열이라 이 차는 대역 선택 하나만 담는다.
2. 고정 대역이 비는 것은 대역 산수가 정한다 — 그 하단이 그 앙각의 f_tip 보다 위라 겹치는 폭이 0 이다.
3. 같은 폭 대역외 기준띠로 견주면 추적 대역은 그 띠 위 +44.60 dB [^37], 고정 대역은 +0.95 dB [^38] 로 띠 높이에 앉는다 — 이 두 여유는 띠를 2.6 kHz 한 자리에 놓아 얻은 값이다.
4. −75° 추적 대역이 담은 것은 f_flash 의 1·2 차 빗살이고, 그 선의 SNR 은 59.2 dB [^39] 와 51.7 dB [^40] 다.
5. −60° 에서는 반송파 몫이 -14.05 dB [^41] 로 내려앉아 «전체 대비» 몫이 혼자 뛴다 — 그래서 반송파 기준 +8.46 dB [^42] 를 나란히 싣는다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 추적 대역 | 0.35~1.0 × f_tip(el) — 앙각마다 다시 잡는다 [^43] |
| 고정 대역 | 430.1~1228.7 Hz 고정 (덱의 −15° 대역) [^44] — 그 상단은 1228.7 Hz [^45] 다 |
| 바닥 잣대 | 대역외 기준띠 — 같은 폭을 2600 Hz 위로 옮긴 것. 블레이드가 원리적으로 못 오는 자리라 «바닥» 을 읽는다 [^46] |
| 몫의 정의 | 대역 전력 / 전체 전력 [dB] — 눈금 무관. verify 의 share_of_total_power_db 와 같은 정의 [^47] |
| 스펙트럼 | 4096 점 한나 창 FFT, Δf = 4.81 Hz (verify_nadir_flash.decompose 와 같은 규약) [^48] |
| 정규화 둘 | 전체 전력 대비 몫과 반송파(동체선) 대비 몫을 칸마다 함께 낸다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/build_ch1_elevation_figs.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/build_part79_normalization_fig.py
```

| | |
|---|---|
| 출력 | `outputs/ch1_elevation_figdata.json`, `outputs/figures/ch1_f4_bandenergy.png`, `outputs/figures/part79_normalization.png` |
| 소요 | 정규화 그림 약 1 초 (CPU). 앙각 그림 다섯 장의 시간은 미측정 |
| 비고 | 두 스크립트 모두 원장을 읽어 CPU 로 FFT 한다[^49] |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 1** «스윕 설계» | 앙각 7 점이 무엇을 어떤 잣대로 쟀고 어느 행을 인용해도 되나 |

---


## 프로펠러 대역은 앙각을 따라 내려간다

블레이드 끝이 만드는 도플러는 **날개끝 주파수 f_tip** 에서 멈춘다 — 기체에서 가장 빠른 산란체가 날개끝이기 때문이다. 그 f_tip 은 관측 앙각을 내리면 함께 내려간다.

우리 팔의 표를 그대로 읽으면 f_tip 은 0° 의 1272.9 Hz [^50] 에서 −75° 의 329.5 Hz [^51] 로 내려가고, −90° 에서 0.0 Hz [^52] 가 된다 — cos(el) 을 곱한 값 그대로다.

그러니 **대역을 어디에 놓느냐**가 판정을 정한다. 여기서는 두 가지로 놓고 잰다.


## 두 대역, 그리고 바닥을 읽는 자

| 대역 | 어디에 놓나 | −75° 에서의 폭 |
|---|---|---|
| 추적 대역 | 0.35~1.0 × f_tip(el) — 앙각마다 다시 잡는다 [^43] | 214 Hz [^53] |
| 고정 대역 | 8/11 덱이 −15° 의 f_tip 으로 못 박은 자리에 그대로 둔다 | 799 Hz [^54] |

두 대역 모두 **자기와 같은 폭의 대역외 기준띠**와 견준다[^46]. 그 띠는 2.6 kHz 위에 있어 블레이드가 원리적으로 못 오는 자리다 — 대역 몫이 그 띠와 같으면 그것은 신호가 아니라 **바닥**이다.


## −75° 한 자리에서 두 대역이 갈린다

| 무엇을 | 대역 몫 | 같은 폭 기준띠 | 띠 위 여유 |
|---|---|---|---|
| 추적 대역, −75° | -10.88 dB [^34] | -55.48 dB [^55] | +44.60 dB [^37] |
| 고정 대역, −75° | -49.60 dB [^35] | -50.55 dB [^56] | +0.95 dB [^38] |
| 고정 대역, −90° | -51.64 dB [^57] | -51.47 dB [^58] | -0.17 dB [^59] |


추적 대역은 −75° 에서 고정 대역보다 좁다. 좁은 대역이 전체 전력의 +38.72 dB [^36] 만큼 더 큰 몫을 가져갔다는 것은 그 에너지가 거기에 **몰려 있다**는 뜻이다. 두 몫의 분모가 같은 시계열이라 이 차는 대역 선택 하나만 담는다.

고정 대역이 비는 것은 대역 산수가 정한다 — 그 하단 430.1 Hz 가 −75° 의 f_tip 329.5 Hz [^51] 보다 위라 겹치는 폭이 0.0000 [^60] 다.


## −75° 의 추적 대역을 채운 것은 저차 빗살이다

f_flash 의 정수배에 선 빗살 선 SNR 을 원장이 30 차까지 싣는다[^61]. −75° 에서 추적 대역(115.3~329.5 Hz)에 드는 차수는 1·2 차이고, 그 두 선은 59.2 dB [^39] 와 51.7 dB [^40] 로 서 있다. 덱 대역이 담는 4~9 차는 같은 칸에서 9.1 dB [^62] ~ 17.3 dB [^63] 다.

f_flash 는 앙각과 무관하고 f_tip 만 cos(el) 로 줄기 때문에, 창이 담는 차수는 앙각이 내려갈수록 낮아진다. −75° 에서 추적 대역이 얻는 이득은 날개끝 신호를 지킨 대가라기보다 창이 가장 센 저차 빗살 위로 내려앉은 결과다.


−90° 는 f_tip 이 0 이라 추적 대역의 폭도 0 이다 — 그 칸은 «측정 불가» 로 비워 뒀다[^64]. 값이 비는 이유는 신호가 약해서가 아니라 **잴 대역이 없어서**다.

같은 −90° 에서 고정 대역 -51.64 dB [^57] 와 그 기준띠 -51.47 dB [^58] 는 -0.17 dB [^59] 안에서 같다. 같다는 것은 두 띠에 **같은 f_flash 빗살이 지나간다**는 뜻이다 — 고정 대역 안 차수 4~9 는 국소 바닥 위 4.4 dB [^65] ~ 24.5 dB [^66], 기준띠 안 차수 21~26 은 8.1 dB [^67] ~ 21.4 dB [^68] 로 서 있다. 이 잣대가 −90° 에서 재는 것은 «대역이 비었나» 라기보다 «블레이드만의 초과분이 있나» 이고, 그 초과분은 0.17 dB 안에서 0 이다.


![ch1_f4_bandenergy](../outputs/figures/ch1_f4_bandenergy.png)

**그림 3.** 앙각을 내리면 프로펠러 대역 에너지가 정말 주는가?

점선이 각 팔의 대역외 기준띠다. **자기 점선 위에 앉은 표식은 바닥에 앉은 것**이고, 오른쪽 패널의 −75°·−90° 에서 붉은 선이 정확히 그렇게 된다.


## 앙각 7 점 전부

| 칸 | f_tip [Hz] | 추적 몫 [dB] | 추적 기준띠 [dB] | 고정 몫 [dB] | 고정 기준띠 [dB] |
|---|---|---|---|---|---|
| ours/el+0 | 1272.9 | -15.58 | -40.18 | -15.59 | -41.13 |
| ours/el-15 | 1229.5 | -21.46 | -36.42 | -21.46 | -36.42 |
| ours/el-30 | 1102.4 | -6.58 | -27.89 | -6.56 | -27.80 |
| ours/el-45 | 900.1 | -10.68 | -35.20 | -10.82 | -34.44 |
| ours/el-60 | 636.5 | -5.58 | -31.11 | -15.31 | -29.61 |
| ours/el-75 | 329.5 | -10.88 | -55.48 | -49.60 | -50.55 |
| ours/el-90 | 0.0 | 측정 불가 | 측정 불가 | -51.64 | -51.47 |

출처 [^69]


추적 대역이 자기 대역외 기준띠 위로 서는 여유는 f_tip 이 살아 있는 여섯 앙각에서 −15° 의 +14.96 dB [^70] 가 가장 좁고 −75° 의 +44.60 dB [^37] 가 가장 넓다. 이 여유는 한 칸 안에서 두 대역의 비라 팔마다 다른 절대 눈금과 무관하다.

같은 여섯 점을 전체 전력 대비 «몫» 으로 보면 −15° 의 -21.46 dB [^71] 와 −60° 의 -5.58 dB [^72] 사이, 폭 15.9 dB [^73] 안에 흩어진다. 그 폭은 잣대를 바꾸면 달라진다 — 기준띠 위 여유로는 29.64 dB, 반송파 기준으로는 28.11 dB 다. 앙각이 내려가는 동안 몫은 단조롭지 않다: −15°, −30°, −45°, −60° 가 각각 -6.58 dB [^74] 와 -10.68 dB [^75] 를 사이에 두고 오르내린다.

고정 대역도 −60° 까지는 자기 기준띠 위로 +14.30 dB [^76] 남아 있다가 −75° 에서 그 띠에 닿는다. 그 지점부터 고정 대역은 f_tip 위에 통째로 놓인다.


## 분모를 바꿔도 같은 결론이 서는가

![part79_normalization](../outputs/figures/part79_normalization.png)

**그림 4.** 전체 전력으로 나눈 몫과 반송파로 나눈 몫이 어디서 갈라지는가?


| 칸 | 반송파 몫 [dB] | 전체 대비 추적 몫 [dB] | 반송파 대비 추적 몫 [dB] |
|---|---|---|---|
| ours/el+0 | -1.90 | -15.58 | -13.69 |
| ours/el-15 | -1.81 | -21.46 | -19.65 |
| ours/el-30 | -3.32 | -6.58 | -3.27 |
| ours/el-45 | -2.43 | -10.68 | -8.25 |
| ours/el-60 | -14.05 | -5.58 | +8.46 |
| ours/el-75 | -2.13 | -10.88 | -8.75 |
| ours/el-90 | -1.76 | 측정 불가 | 측정 불가 |

출처 [^69]


몫의 분모는 **전체 전력**이고, 그 전력의 대부분은 동체선(반송파)이다 — 우리 팔의 반송파 몫은 −60° 를 뺀 여섯 점에서 -1.81 dB [^77] ~ -3.32 dB [^78] 안에 있다. 그래서 반송파가 내려앉는 칸에서는 분자가 그대로여도 몫이 올라간다. −60° 가 그 자리다 — 반송파 몫이 -14.05 dB [^41] 로 내려앉고, 전체 대비 추적 몫 -5.58 dB [^72] 가 우리 팔 여섯 점 중 가장 커진다.


이 내려앉음은 **우리 팔 한 칸의 성질이다.** 같은 −60° 에서 PathSolver 11.1M 은 -3.13 dB [^79], 250M 은 -2.10 dB [^80] 로 다른 앙각과 같다.

정규화를 반송파로 바꾸는 것은 이 칸을 되돌려 놓지 못한다 — 전체 전력의 대부분이 반송파라 두 분모가 함께 내려앉기 때문이다. 반송파 기준으로 재면 −60° 는 +8.46 dB [^42] 로 우리 팔 여섯 점 중 유일한 양수가 되어 오히려 더 튄다.

분모가 약분되는 잣대는 **같은 폭의 대역외 기준띠와의 여유**다. 그 잣대에서 −60° 는 +25.52 dB [^81] 로 0° 의 +24.60 dB [^82] · −45° 의 +24.52 dB [^83] 와 나란하고, −75° 가 +44.60 dB [^37] 로 가장 높다. 헤드라인 자리인 −75° 는 반송파 몫이 -2.13 dB [^84] 로 통상값 안에 있어, 전체 대비와 반송파 대비가 여섯 점 중 같은 순위를 준다[^85].


## 대역에 무언가 남는 것과 프로펠러가 보이는 것은 다른 일이다

| 칸 | 고정 몫 [dB] | 기준띠 [dB] | f_tip [Hz] |
|---|---|---|---|
| ours/el-75 | -49.60 | -50.55 | 329.5 |
| sionna/el-75 | -10.58 | -23.50 | 329.5 |
| sionna_p250000000/el-75 | -22.04 | -36.06 | 329.5 |
| ours/el-90 | -51.64 | -51.47 | 0.0 |
| sionna/el-90 | -7.16 | -20.20 | 0.0 |
| sionna_p250000000/el-90 | -16.92 | -32.87 | 0.0 |

출처 [^69]


−75° 에서 고정 대역은 f_tip = 329.5 Hz [^51] 위에 통째로 놓인다 — 겹치는 폭이 0.0000 [^60] 다. 우리 팔은 그 자리를 기준띠 높이로 비워 두고, PathSolver 팔들은 +12.93 dB [^86] 와 +14.03 dB [^87] 를 채워 넣는다.


⭐그 세 숫자가 재는 범위는 여기까지다. 기준띠에도 빗살 21~26 차가 들어 있어서, 이 여유는 «빗살 든 칸 대 빈 칸» 이 아니라 **빗살 대 빗살**의 대비다 — 같은 −75° 에서 11.1M 팔의 고정 대역 차수 4~9 는 국소 바닥 위 31.5 dB [^88] ~ 39.7 dB [^89] 이고, 기준띠 차수 21~26 은 15.1 dB [^90] ~ 23.3 dB [^91] 다. 빗살의 간격은 회전 입력 f_flash 가 정하므로, 빗살이 섰다는 사실은 산란 커널의 점수와 다른 물건이다.

가르는 것은 대역을 f_tip 따라 옮겼을 때다 — 같은 −75° 에서 우리 팔의 추적 대역은 자기 기준띠보다 +44.60 dB [^37] 높다. 빗살이 f_tip 위에서도 살아남는다는 것은 관측 사실이고, 그 빗살이 블레이드 도플러인지 자세별 경로 집합의 깜빡임인지는 **절 5** «광선 예산» 이 예산 축에서 가른다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 검출기의 도플러 대역을 앙각 추정값의 f_tip 에 묶는다 | −75° 이하에서 고정 대역이 잃는 여유를 추적 대역이 되찾는지가 검출 확률로 결정된다 | `src/passive_process.py` 의 대역 설정 |
| −75° 와 −90° 사이에 앙각 한 점을 더 잰다 | 추적 대역의 폭이 언제 0 으로 닫히는지가 관측으로 정해진다 | `benchmark/elevation_sweep_md.py --els -82.5` |
| 기하 겹침 예측을 이 두 대역 위에서 다시 채점한다 | «대역이 겹치는 만큼 에너지가 준다» 는 예측의 성립 범위가 확정된다 | `outputs/ch1_elevation_figdata.json` 의 `prediction.fixed_band_overlap_frac` |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 58개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^34] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.share_track_db` | -10.88 |
| [^35] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.share_fixed_db` | -49.6 |
| [^36] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.share_track_db → 같은 칸의 share_fixed_db 를 뺀 값` | -10.88 (파생) |
| [^37] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.share_track_db → 같은 칸의 share_track_oob_db 를 뺀 값` | -10.88 (파생) |
| [^38] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.share_fixed_db → 같은 칸의 share_fixed_oob_db 를 뺀 값` | -49.6 (파생) |
| [^39] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.comb_snr_db[0]` | 59.2 |
| [^40] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.comb_snr_db[1]` | 51.7 |
| [^41] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-60.carrier_share_db` | -14.05 |
| [^42] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-60.share_track_rel_carrier_db` | 8.46 |
| [^43] | `outputs/ch1_elevation_figdata.json` | `_meta.band_track_ko` | 0.35~1.0 × f_tip(el) — 앙각마다 다시 잡는다 |
| [^44] | `outputs/ch1_elevation_figdata.json` | `_meta.band_fixed_ko` | 430.1~1228.7 Hz 고정 (덱의 −15° 대역) |
| [^45] | `outputs/report07_three_engines.json` | `_meta.f_tip_hz` | 1229 |
| [^46] | `outputs/ch1_elevation_figdata.json` | `_meta.oob_ko` | 대역외 기준띠 — 같은 폭을 2600 Hz 위로 옮긴 것. 블레이드가 원리적으로 못 오는 자리라 «… |
| [^47] | `outputs/ch1_elevation_figdata.json` | `_meta.share_ko` | 대역 전력 / 전체 전력 [dB] — 눈금 무관. verify 의 share_of_total_pow… |
| [^48] | `outputs/ch1_elevation_figdata.json` | `_meta.spectrum_ko` | 4096 점 한나 창 FFT, Δf = 4.81 Hz (verify_nadir_flash.decom… |
| [^49] | `outputs/ch1_elevation_figdata.json` | `_meta.gpu_ko` | GPU 를 쓰지 않았다. 원장만 읽고 CPU 로 FFT 했다. |
| [^50] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el+0.f_tip_hz` | 1273 |
| [^51] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.f_tip_hz` | 329.5 |
| [^52] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.f_tip_hz` | 0 |
| [^53] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.f_tip_hz → 0.65 를 곱한 추적 대역 폭` | 329.5 (파생) |
| [^54] | `outputs/report07_three_engines.json` | `_meta.f_tip_hz → 0.65 를 곱한 고정 대역 폭` | 1229 (파생) |
| [^55] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.share_track_oob_db` | -55.48 |
| [^56] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.share_fixed_oob_db` | -50.55 |
| [^57] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.share_fixed_db` | -51.64 |
| [^58] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.share_fixed_oob_db` | -51.47 |
| [^59] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.share_fixed_db → 같은 칸의 share_fixed_oob_db 를 뺀 값` | -51.64 (파생) |
| [^60] | `outputs/ch1_elevation_figdata.json` | `prediction.fixed_band_overlap_frac.-75` | 0 |
| [^61] | `outputs/ch1_elevation_figdata.json` | `_meta.comb_ko` | 빗살 선 SNR = k·f_flash ±12 Hz 전력 / 그 옆 30~70 Hz 중앙값 |
| [^62] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.comb_snr_db[7]` | 9.1 |
| [^63] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.comb_snr_db[6]` | 17.3 |
| [^64] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.share_track_db` | null |
| [^65] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.comb_snr_db[3]` | 4.4 |
| [^66] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.comb_snr_db[6]` | 24.5 |
| [^67] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.comb_snr_db[23]` | 8.1 |
| [^68] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.comb_snr_db[20]` | 21.4 |
| [^69] | `outputs/ch1_elevation_figdata.json` | `cells` | (21항목 묶음) |
| [^70] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-15.share_track_db → 같은 칸의 share_track_oob_db 를 뺀 값` | -21.46 (파생) |
| [^71] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-15.share_track_db` | -21.46 |
| [^72] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-60.share_track_db` | -5.584 |
| [^73] | `outputs/ch1_elevation_figdata.json` | `gates.G6_ours_track_share_span_db` | 15.9 |
| [^74] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-30.share_track_db` | -6.584 |
| [^75] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-45.share_track_db` | -10.68 |
| [^76] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-60.share_fixed_db → 같은 칸의 share_fixed_oob_db 를 뺀 값` | -15.31 (파생) |
| [^77] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-15.carrier_share_db` | -1.81 |
| [^78] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-30.carrier_share_db` | -3.32 |
| [^79] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el-60.carrier_share_db` | -3.13 |
| [^80] | `outputs/ch1_elevation_figdata.json` | `cells.sionna_p250000000/el-60.carrier_share_db` | -2.1 |
| [^81] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-60.share_track_db → 같은 칸의 share_track_oob_db 를 뺀 값` | -5.584 (파생) |
| [^82] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el+0.share_track_db → 같은 칸의 share_track_oob_db 를 뺀 값` | -15.58 (파생) |
| [^83] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-45.share_track_db → 같은 칸의 share_track_oob_db 를 뺀 값` | -10.68 (파생) |
| [^84] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.carrier_share_db` | -2.13 |
| [^85] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-75.share_track_rel_carrier_db` | -8.75 |
| [^86] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el-75.share_fixed_db → 같은 칸의 share_fixed_oob_db 를 뺀 값` | -10.58 (파생) |
| [^87] | `outputs/ch1_elevation_figdata.json` | `cells.sionna_p250000000/el-75.share_fixed_db → 같은 칸의 share_fixed_oob_db 를 뺀 값` | -22.04 (파생) |
| [^88] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el-75.comb_snr_db[7]` | 31.5 |
| [^89] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el-75.comb_snr_db[3]` | 39.7 |
| [^90] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el-75.comb_snr_db[24]` | 15.1 |
| [^91] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el-75.comb_snr_db[21]` | 23.3 |


---

## 절 3. 물리 상한 위 누설은 우리 팔 0.26~15.65 %, 스톡 PathSolver 두 예산 5.12~87.02 % 이고, 그 위 끝은 평평한 스펙트럼이 받는 점수다



> ### 한 일
> **날개끝 속도가 정하는 상한 위 대역의 에너지 몫을 세 팔 여섯 앙각에서 재고 엔진을 가르는 잣대로 삼았다.**

### 결과
1. 상한은 f_tip = 1272.9 Hz [^92] × cos(el) 이고 관찰 상한은 나이퀴스트 9850 Hz [^93] 다 — el 0° 에서 순시 도플러가 닿는 폭은 관찰 대역의 87.08% [^94] 를 남긴다.
2. 우리 팔의 누설은 el −75° 의 0.26% [^95] 에서 el −15° 의 15.65% [^96] 사이다 — f_tip 이 0 보다 큰 여섯 앙각의 값이다.
3. PathSolver 두 예산은 5.12% [^97] ~ 87.02% [^98] 로, 양 끝이 모두 250M 팔에서 나온다.
4. 여섯 앙각 중 다섯에서 우리 팔이 두 예산 모두보다 낮고, el −30° 에서는 우리 11.17% [^99] 가 250M 의 5.12% [^97] 보다 높다.
5. 이 잣대의 위 끝은 포화점이다 — el 0° 에서 평평한 스펙트럼이 받는 점수가 87.08% [^94] 이고 250M 팔의 87.02% [^98] 가 그 값이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 상한의 정의 | f_tip = 2·(2π f_rev R)/λ · cos(el) — `benchmark/elevation_sweep_md.py:205` 의 `f_tip_at()` 한 곳에 있다 |
| 표적의 운동 | 기체는 10 m 한 자리에 고정하고 회전자 위상만 시간에 따라 돌린다 — `benchmark/elevation_sweep_md.py:105` 의 `rotor_phases()` |
| 누설 몫 | 자세 시계열의 슬로타임 FFT(한나 창)에서 \|f\| ≥ f_tip 전력 ÷ 전체 전력 — `benchmark/build_wideband_energy_fig.py:133` |
| 정규화 | 팔마다 자기 전체 전력으로 나눈 몫이라 팔 사이 절대 레벨은 이 표 밖이다 |
| 행 선택 | `n_missing = 0` 인 행만 쓴다 — 원장이 부분 병합 행을 미리 뺐다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/build_wideband_energy_fig.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/build_above_tip_fig.py
```

| | |
|---|---|
| 출력 | `outputs/wideband_energy.json`, `outputs/figures/ch1_f6_above_tip.png` |
| 소요 | 그림 4.95 초 (CPU 전용) · 원장 재생성은 미측정 |
| 비고 | 시계열은 `outputs/elevation_sweep_md.npz` 에서 읽는다. 팔의 완결성은 `outputs/elevation_sweep_md.json` 의 `n_missing` 이 정한다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 1** «스윕 규약» | 이 스윕이 무엇을 어떤 규약으로 쟀고 어느 행을 인용해도 되나 |
| **절 2** «대역 추적» | 대역을 앙각마다 f_tip 으로 옮겨 잡는 규약 |

---


## 상한은 날개끝 속도가 긋는다

회전자 날개끝이 시선 방향으로 내는 속도가 **순시 도플러**의 상한을 정한다. 그 상한이 f_tip 이고, 앙각이 내려가면 날개 속도가 시선에 수직해지면서 cos(el) 로 줄어든다 — el 0° 에서 1272.9 Hz [^100], 직하방에서 0.0 Hz [^101] 다.

관찰 상한은 그와 무관하게 표본율이 정한다 — PRF 19700 Hz [^102] 의 절반인 나이퀴스트 9850 Hz [^93] 다.

⇒ el 0° 에서 순시 도플러가 닿는 폭 밖에 남는 자리는 관찰 대역의 87.08% [^94] 이고, 직하방에서는 100.00% [^103] 다. 원장은 그 자리를 인공물로 적는다[^104].


그 «인공물» 을 0 이 참값인 자리로 읽는 것은 이 잣대가 서는 범위 **밖**이다. f_tip 은 순시 주파수의 상한이지 스펙트럼이 끊기는 자리가 아니다 — 회전 날개의 위상변조 측대역은 그 위로 이어진다. 그래서 이상적인 날개도 상한 위에 작은 양수를 남긴다.

상한선의 정밀도도 여기까지다. f_tip 은 호버 회전수 한 값으로 긋는데 이 스윕의 네 로터는 «덱과 같은 결정론 패턴(OU 프리셋 아님) — 축을 하나만 바꾼다 [^105]» 로 돌아, 가장 빠른 로터의 순시 상한은 el 0° 에서 f_tip 보다 몇 Hz 위에 있다.

⇒ 이 절은 상한 위 몫을 **같은 앙각에서 팔끼리 견주는 상대 잣대**로 쓴다. 바닥이 0 인 절대 눈금으로 올리려면 진폭변조를 포함한 참조 신호로 물리 꼬리를 먼저 빼야 하고, 그 빼기는 다음 단계 표의 첫 줄이다.


### 이 절의 세팅

| 설정 | 값 |
|---|---|
| 표적 | matrice4e [^106] |
| 거리 | 10.0 m [^107] |
| PRF | 19700 Hz [^102] |
| 자세 표본 | 4096 개 [^108] |
| 격자 | 얼린 격자(자세 합집합 bbox), λ/12 [^109] |
| 빠진 행 | n_missing > 0 인 행은 제외했다(부분 병합은 시계열에 0 이 박힌다). [^110] |


![above tip limit](../outputs/figures/ch1_f6_above_tip.png)

**그림 5.** 물리적으로 블레이드가 닿을 수 없는 대역에 어느 팔이 얼마를 남기는가?

(a) 는 상한이 앙각을 따라 어디에 그어지는지, (b) 는 그 위에 팔마다 얼마가 놓이는지다. (b) 의 모든 막대는 참값이 0 이다.


## 세 팔이 상한 위에 남기는 몫

| 앙각 | f_tip | 우리 커널 | PathSolver 11.1M | PathSolver 250M |
|---|---|---|---|---|
| +0° | 1272.9 Hz [^100] | 2.28% [^111] | 29.03% [^112] | 87.02% [^98] |
| -15° | 1229.5 Hz [^113] | 15.65% [^96] | 40.04% [^114] | 30.87% [^115] |
| -30° | 1102.4 Hz [^116] | 11.17% [^99] | 8.81% [^117] | 5.12% [^97] |
| -45° | 900.1 Hz [^118] | 2.18% [^119] | 25.30% [^120] | 7.39% [^121] |
| -60° | 636.5 Hz [^122] | 1.34% [^123] | 28.46% [^124] | 18.53% [^125] |
| -75° | 329.5 Hz [^126] | 0.26% [^95] | 47.08% [^127] | 5.83% [^128] |


## 이 잣대가 대역 안 에너지와 다른 점

대역 안(`500 Hz` 부터 f_tip 까지) 몫은 el 0° 에서 우리 -0.50 dB [^129], 11.1M -1.59 dB [^130], 250M -11.04 dB [^131] 로 갈린다. 셋 중 무엇이 참인지는 표적의 산란과 조명 기하가 정하고, 그 참값은 이 원장 밖에 있다.

상한 위의 참값도 같은 것이 정한다 — 날개 위 진폭 분포가 위상변조 꼬리의 크기를 정하기 때문이다. 두 잣대의 참값은 같은 이유로 원장 밖에 있다.

상한 위가 더 강한 것은 **그 참값이 작다고 알려져 있다**는 데 있다. 대역 안 몫은 0.2~0.9 사이 어디든 될 수 있지만, 상한 위 몫은 이상적 날개에서 한 자릿수 퍼센트 아래로 묶인다. 그래서 이 잣대는 큰 값을 인공물로 읽는 데 쓰고, 작은 값들 사이의 순위는 이 절의 범위 밖이다.


### 이 잣대가 서는 범위

| 조건 | 이 절이 재는 자리 |
|---|---|
| 관찰 상한 | 9850 Hz [^93] 아래만 읽는다 — 그 위 성분은 접혀 들어와 대역 안에 더해진다 |
| 표적의 운동 | 기체 고정 · 회전자만 회전. 기체가 움직이면 상한이 동체 도플러만큼 옮겨간다 |
| 앙각 | f_tip > 0 인 여섯 앙각. 직하방은 f_tip = 0 이라 몫이 정의되지 않는다 |
| 포화점 | 평평한 스펙트럼이 받는 점수 — el 0° 에서 87.08% [^94], el −75° 에서 96.65% [^132]. 그 값에 닿은 팔들 사이의 순위는 읽지 않는다 |
| 비교 축 | 같은 앙각에서 팔끼리 몫을 견준다. 앙각을 가로지르는 비교는 분모가 달라 이 표 밖이다 |
| 격자 | 우리 팔은 λ/12 한 판이다. 같은 앙각에서 격자만 바꾸면 이 잣대가 1.37% [^133] 에서 0.085% [^134] 로 움직인다(다른 추정기·같은 방향) |
| 합격선 | 미정 — 임계는 다음 단계 표가 정한다 |


## 우리 팔도 −15°·−30° 두 자리에서 샌다

우리 팔의 누설은 네 앙각에서 0.26% [^95] ~ 2.28% [^111] 이고, el −15° 에서 15.65% [^96], el −30° 에서 11.17% [^99] 로 올라간다.

el −30° 는 이 표의 세 팔 중 우리 팔이 가장 높은 유일한 자리다 — 250M 의 5.12% [^97] 와 11.1M 의 8.81% [^117] 위다. 같은 잣대가 우리 팔에도 인공물을 두 자리에서 드러낸다.

그 순위는 엔진의 순위라기보다 **표본화 밀도의 순위**다. 우리 팔의 판은 앙각마다 λ/12 하나뿐인데, 같은 표적·같은 el −15° 에서 격자만 λ/8 → λ/32 로 바꾸면 같은 종류의 대역밖 몫이 1.37% [^133] 에서 0.085% [^134] 로 16 배 내려간다(얼린 격자 팔, 블레이드 대역을 분모로 쓰는 다른 추정기). PathSolver 쪽에서는 광선 예산이 같은 일을 한다 — el −30° 에서 8.81% [^117] → 5.12% [^97] 다.

원인 후보는 셋이다 — λ/12 격자의 표본화, 10 m 근접장 곡률, 자세 시계열 창의 누설. 셋을 가르는 실험은 다음 단계 표에 있고, 나딧에서 같은 삼분할을 실제로 수행한 선례가 **절 4** «나딧 잔여» 다. 이 절이 확정한 것은 두 자리의 크기까지다.


## 광선을 늘리면 다섯 자리에서 줄고 한 자리에서 는다

11.1M 과 250M 은 광선 수만 다른 두 판이다 — 스윕이 `--spp` 하나로 가른다(`benchmark/elevation_sweep_md.py:339`).

el 0° 에서 누설은 29.03% [^112] → 87.02% [^98] 로 늘고, 나머지 다섯 앙각에서는 줄어든다 — el −75° 는 47.08% [^127] → 5.83% [^128] 다.

⇒ 광선 예산은 이 누설을 앙각마다 다른 방향으로 옮긴다. 예산 축 자체는 **절 5** «광선 예산» 가 잰다.


이 표의 범위는 **물리 스위치를 끈 두 팔**이다. 굴절·회절·모서리회절과 다중반사를 켠 팔은 같은 잣대에서 더 샌다 — 11.1M + 물리는 el −45° 의 46.09% [^135] 에서 el −60° 의 88.01% [^136] 사이이고, 250M + 물리는 el −45° 의 79.48% [^137] 에서 el 0° 의 87.11% [^138] 사이다[^139]. 물리 팔의 판정은 [리포트 17 절 3 «물리 팔의 상한 위 누설»](17_engine-physics.ipynb) 가 맡는다.


### 검증된 것과 열어 둔 것

| 무엇 | 상태 |
|---|---|
| 상한 f_tip(el) 의 위치 | 운동학 정의 — 원장이 앙각마다 값을 싣는다 |
| 세 팔 18 칸의 누설 몫 | 계산값 — 합격선은 다음 단계가 정한다 |
| 우리 팔 두 자리의 원인 | 열린 과제 — 후보 셋을 다음 단계가 가른다 |


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 이상적 회전 블레이드의 상한 위 꼬리 τ 를 같은 규약으로 계산해 원장에 싣는다 | 이 잣대의 바닥이 0 에서 τ 로 바뀌고, 우리 팔의 작은 값들이 τ 안인지 밖인지 갈린다 | `outputs/tip_tail_reference.json` 신규 (CPU 수 초) · **새 계산이 필요하다** |
| 상한 위 누설의 합격선을 τ 위에서 정해 게이트로 올린다 | 팔을 통과·불통과로 가르는 잣대가 하나 늘고 이 절의 표가 판정표가 된다 | `benchmark/build_wideband_energy_fig.py` 에 임계와 판정 칸 추가 |
| 우리 팔 el −15°·−30° 를 λ/48 격자로 다시 돌린다 | 그 두 자리의 누설이 격자 표본화에서 오는지가 수치로 갈린다 | `benchmark/elevation_sweep_md.py:76` 의 `DIV = 12` → 48 · **새 계산이 필요하다** |
| 디스크에 있는 물리 팔 샤드를 병합해 앙각 칸을 채운다 | 물리 축이 이 누설을 어느 쪽으로 옮기는지가 두 점에서 네 점으로 는다 | `benchmark/elevation_sweep_md.py --merge` — 기존 원장을 덮어쓴다 |
| PRF 를 올려 나이퀴스트 위에서 접혀 들어오는 몫을 잰다 | 관찰 상한이 이 잣대에 넣는 몫이 갈린다 | `benchmark/elevation_sweep_md.py` PRF 축 · **새 계산이 필요하다** |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 48개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^92] | `outputs/wideband_energy.json` | `_meta.f_tip_el0_hz` | 1273 |
| [^93] | `outputs/wideband_energy.json` | `_meta.nyquist_hz` | 9850 |
| [^94] | `outputs/wideband_energy.json` | `_meta.nyquist_hz → cells.ours/el+0.f_tip_hz 를 빼고 나눈 값` | 9850 (파생) |
| [^95] | `outputs/wideband_energy.json` | `cells.ours/el-75.above_f_tip_frac` | 0.00255 |
| [^96] | `outputs/wideband_energy.json` | `cells.ours/el-15.above_f_tip_frac` | 0.1565 |
| [^97] | `outputs/wideband_energy.json` | `cells.sionna_p250000000/el-30.above_f_tip_frac` | 0.05121 |
| [^98] | `outputs/wideband_energy.json` | `cells.sionna_p250000000/el+0.above_f_tip_frac` | 0.8702 |
| [^99] | `outputs/wideband_energy.json` | `cells.ours/el-30.above_f_tip_frac` | 0.1117 |
| [^100] | `outputs/wideband_energy.json` | `cells.ours/el+0.f_tip_hz` | 1273 |
| [^101] | `outputs/wideband_energy.json` | `cells.ours/el-90.f_tip_hz` | 0 |
| [^102] | `outputs/wideband_energy.json` | `_meta.prf_hz` | 19700 |
| [^103] | `outputs/wideband_energy.json` | `_meta.nyquist_hz → cells.ours/el-90.f_tip_hz 를 빼고 나눈 값` | 9850 (파생) |
| [^104] | `outputs/wideband_energy.json` | `_meta.physical_limit_ko` | f_tip = 1272.9·cos(el) 가 날개끝 속도가 정하는 상한이다. 그 위의 에너지는 블레… |
| [^105] | `outputs/elevation_sweep_md.json` | `_meta.rotor_ko` | 덱과 같은 결정론 패턴(OU 프리셋 아님) — 축을 하나만 바꾼다 |
| [^106] | `outputs/elevation_sweep_md.json` | `_meta.drone` | matrice4e |
| [^107] | `outputs/elevation_sweep_md.json` | `_meta.range_m` | 10 |
| [^108] | `outputs/elevation_sweep_md.json` | `rows[0].n_poses` | 4096 |
| [^109] | `outputs/elevation_sweep_md.json` | `_meta.grid_ko` | 얼린 격자(자세 합집합 bbox), λ/12 |
| [^110] | `outputs/wideband_energy.json` | `_meta.incomplete_excluded_ko` | n_missing > 0 인 행은 제외했다(부분 병합은 시계열에 0 이 박힌다). |
| [^111] | `outputs/wideband_energy.json` | `cells.ours/el+0.above_f_tip_frac` | 0.02285 |
| [^112] | `outputs/wideband_energy.json` | `cells.sionna/el+0.above_f_tip_frac` | 0.2903 |
| [^113] | `outputs/wideband_energy.json` | `cells.ours/el-15.f_tip_hz` | 1230 |
| [^114] | `outputs/wideband_energy.json` | `cells.sionna/el-15.above_f_tip_frac` | 0.4004 |
| [^115] | `outputs/wideband_energy.json` | `cells.sionna_p250000000/el-15.above_f_tip_frac` | 0.3087 |
| [^116] | `outputs/wideband_energy.json` | `cells.ours/el-30.f_tip_hz` | 1102 |
| [^117] | `outputs/wideband_energy.json` | `cells.sionna/el-30.above_f_tip_frac` | 0.08809 |
| [^118] | `outputs/wideband_energy.json` | `cells.ours/el-45.f_tip_hz` | 900.1 |
| [^119] | `outputs/wideband_energy.json` | `cells.ours/el-45.above_f_tip_frac` | 0.02181 |
| [^120] | `outputs/wideband_energy.json` | `cells.sionna/el-45.above_f_tip_frac` | 0.253 |
| [^121] | `outputs/wideband_energy.json` | `cells.sionna_p250000000/el-45.above_f_tip_frac` | 0.07388 |
| [^122] | `outputs/wideband_energy.json` | `cells.ours/el-60.f_tip_hz` | 636.5 |
| [^123] | `outputs/wideband_energy.json` | `cells.ours/el-60.above_f_tip_frac` | 0.0134 |
| [^124] | `outputs/wideband_energy.json` | `cells.sionna/el-60.above_f_tip_frac` | 0.2846 |
| [^125] | `outputs/wideband_energy.json` | `cells.sionna_p250000000/el-60.above_f_tip_frac` | 0.1853 |
| [^126] | `outputs/wideband_energy.json` | `cells.ours/el-75.f_tip_hz` | 329.5 |
| [^127] | `outputs/wideband_energy.json` | `cells.sionna/el-75.above_f_tip_frac` | 0.4708 |
| [^128] | `outputs/wideband_energy.json` | `cells.sionna_p250000000/el-75.above_f_tip_frac` | 0.05831 |
| [^129] | `outputs/wideband_energy.json` | `cells.ours/el+0.500-f_tip` | -0.5 |
| [^130] | `outputs/wideband_energy.json` | `cells.sionna/el+0.500-f_tip` | -1.59 |
| [^131] | `outputs/wideband_energy.json` | `cells.sionna_p250000000/el+0.500-f_tip` | -11.04 |
| [^132] | `outputs/wideband_energy.json` | `_meta.nyquist_hz → cells.ours/el-75.f_tip_hz 를 빼고 나눈 값` | 9850 (파생) |
| [^133] | `outputs/sbr_grid_convergence.json` | `rows[0].froz_frac_power_beyond_ftip` | 0.01371 |
| [^134] | `outputs/sbr_grid_convergence.json` | `rows[4].froz_frac_power_beyond_ftip` | 0.0008541 |
| [^135] | `outputs/wideband_energy_fairbudget.json` | `cells.sionna_phys/el-45.above_f_tip_frac` | 0.4609 |
| [^136] | `outputs/wideband_energy_fairbudget.json` | `cells.sionna_phys/el-60.above_f_tip_frac` | 0.8801 |
| [^137] | `outputs/wideband_energy_fairbudget.json` | `cells.sionna_p250000000_phys/el-45.above_f_tip_frac` | 0.7948 |
| [^138] | `outputs/wideband_energy_fairbudget.json` | `cells.sionna_p250000000_phys/el+0.above_f_tip_frac` | 0.8711 |
| [^139] | `outputs/wideband_energy_fairbudget.json` | `selftest.verdict` | PASS |


---

## 절 4. 나딧 잔여 −38.31 dB 의 64 % 는 광선 격자 표본화 잡음이고, 5° 만 기울면 −11.88 dB 로 열린다



> ### 한 일
> **직하방에서 남은 변조를 세 갈래로 귀속하고, 나딧에서 벗어난 각도별 변조 예산과 거리 거동을 원장에서 뽑았다.**

### 결과
1. 원거리장 나딧의 변조는 -305.01 dB [^140] — 회전이 산란적분을 그대로 두는 자리다.
2. 10 m 판에 남는 변조는 -38.31 dB [^141] 이고, 그 전력의 63.7% [^142] 가 광선 격자 표본화 잡음이다.
3. 나딧에서 0.5° [^143] 벗어나면 -55.85 dB [^144], 5° [^145] 에서 -11.88 dB [^146] 로 변조가 돌아온다 — 사각지대는 좁은 원뿔이다.
4. 생산 격자에서 재면 이 잔여는 10 m -48.46 dB [^147] 에서 1 km -50.31 dB [^148] 까지 평평하다.
5. 나딧의 몸체 에코는 오히려 커진다 — 10 m SNR 44.39 dB [^149] 대 −15° 의 34.91 dB [^150] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 변조 잣대 | 자세 시계열의 AC/DC [dB] — 반송파(DC) 대비 흔들리는 전력. `benchmark/verify_nadir_flash.py` 의 분해 규약을 그대로 쓴다 |
| 삼분할 | 커널을 CPU 로 재현해 격자·근접장·가림을 한 축씩 갈라 남는 전력의 몫으로 나눈다. 세 갈래가 직교하지 않으므로 이 나눗셈은 어림이다 |
| 각도 예산 | 재질·가림·격자 없는 평면 패싯 PO 대리모형의 원거리장 값이다. 각도 의존만 읽고 절대 레벨은 SBR 원장 쪽을 쓴다 |
| 판 | matrice4e 1기 · 3.5 GHz · 거리 10 m [^151] · 방위 고정 · el −90 행의 자세 4096 개 [^152] (결측 0 개 [^153]) |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_nadir_flash.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/refute_nadir_mechanism_final.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/build_part12_elevation.py
```

| | |
|---|---|
| 출력 | `outputs/verify_nadir_flash.json`, `outputs/refute_nadir_mechanism_final.json` |
| 소요 | 약 2분 (CPU) |
| 비고 | 두 스크립트 모두 GPU 를 쓰지 않는다 — 광선은 CPU z-버퍼이고 mitsuba·sionna 를 import 하지 않는다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 1** «스윕 규약» | 이 스윕이 무엇을 어떤 규약으로 쟀고 어느 행을 인용해도 되나 |
| **절 3** «상한 위 누설» | 날개끝 상한 위 에너지가 왜 인공물의 눈금인가 |

---


## 원거리장 나딧에서 회전은 산란적분을 그대로 둔다

시선이 로터 면에 수직이면 날개의 속도가 시선과 직각이라 도플러 항이 0 이 된다. 평면파로 계산한 나딧 변조 -305.01 dB [^140] 는 배정밀도 반올림의 크기다 — 회전이 시선 방향 좌표도 투영 면적도 바꾸지 않아 적분값이 자세에 불변이기 때문이다.

근거는 원장의 한 줄이다 — `⭐나딧 원거리장에서는 회전이 z 를 바꾸지 않고 투영면적도 바꾸지 않으므로 ∫exp(j2kz)dA_proj 가 **정확히 불변**이다 — 날개 모양·피치·개수와 무관하다. 위 −300 dB 대는 float64 반올림이다.`


## 10 m 판에서는 굽은 파면이 두 날개를 다르게 본다

이 스윕의 거리 10 m 는 원거리장 경계 안쪽이다 — 원장이 판 조건에 그 사실을 적어 뒀다: `⭐사용자 지시로 10 m 고정. ⚠원거리장 경계 2D²/λ ≈ 14.08 m 의 **안쪽**이라 근거리장 판이다 — 우리 커널은 range_m 구면파로 처리하고 PathSolver 는 실제 기하라 둘 다 다룰 수 있지만, 평면파 원거리장 값과 직접 비교하면 안 된다.`

굽은 파면에서는 허브가 회전축에서 0.2196 m [^154] 밀려 있는 만큼 날개끝 왕복거리가 흔들리고, 그 흔들림은 왕복 위상 50.55° [^155] peak-peak 다. 2엽 로터는 그 1차 항을 상쇄하고 2ψ 성분만 남기므로 해석 예측은 -40.61 dB [^156] 이고, 관측 -38.31 dB [^141] 가 그 옆에 선다.

이 예측을 어디까지 읽어도 되는지는 원장이 적어 뒀다 — `해석 예측 -40.6 dB 대 관측 -38.31 dB — **2 dB 안**이다. 다만 이것은 «로터 기여만이 전부» 라고 놓은 값이라, 정확한 예측이 아니라 **크기의 눈금**으로 읽어야 한다.`


## 남은 변조의 3 분의 2 는 수치 인공물이다

![ch1_nadir_residual](../outputs/figures/ch1_nadir_residual.png)

**그림 6.** 나딧에 남은 변조는 실물 표적의 신호인가?

| 갈래 | AC 전력 몫 | 무엇인가 |
|---|---|---|
| 광선 격자 표본화 잡음 | 63.7% [^142] | 격자가 두 날개를 다르게 표본화해 생기는 수치 인공물 — 실물 표적에는 없다 |
| 근접장 파면 곡률 | 31.1% [^157] | 10 m 에서만 사는 기하 항 — 물리적으로는 거리에 1/r⁴ 로 준다 |
| 가림 | 5.2% [^158] | 날개가 몸체를 스치며 가리는 몫 — 조인 격자 판 값이라 상한이다 |


### 이 나눗셈이 서는 범위

가림 몫이 상한이므로 격자 잡음 몫은 하한이다. 근거는 원장에 한 줄로 있다 — `R4c: λ/12 총합 − 근접장고유(coherent diff) − 가림바닥(λ/48 plane, 상한). 측정 AC 는 재현기 AC 의 3.2배 스칼라이므로(ρ=0.998) 같은 비율로 나뉜다고 본다.`

재현기가 관측을 되살린다는 근거는 상관 0.998 [^159] 이고, 같은 계열을 시간축으로 밀어 만든 널 분포의 99 백분위는 0.4618 [^160] 다.


## 이 잔여는 거리를 늘려도 2 dB 안에서 평평하다

생산 격자에서 근접장 고유항만 남기면 거리 지수는 2.589 [^161] 이고, 격자를 네 배 조이면 3.74 [^162] 로 4 에 다가간다. 총 AC/DC 는 10 m -48.46 dB [^147] 에서 1 km -50.31 dB [^148] 까지 평평하다.

거리 지수 4.038 [^163] 는 격자·재질·가림이 없는 패싯 대리모형의 성질이다. 그림 6 오른쪽 판이 두 거동을 나란히 놓는다 — 실전 거리에서 이 바닥이 얼마나 낮은지는 **거리가 아니라 격자**가 정한다.


## 나딧에서 고정 대역이 보고하는 박자는 창 누설이다

70 표본 한나 창의 주엽 반폭 562.9 Hz [^164] 가 대역 하단 430.1 Hz [^165] 를 덮는다. 그래서 el −90 에서 그 대역이 보고하는 전력 -55.68 dB [^166] 는 누설분만 넣었을 때의 값과 0.01 dB [^167] 차이다.

전 구간 FFT(`4096 점 한나 창 FFT, Δf = 4.81 Hz (verify_nadir_flash.decompose 와 같은 규약)`)로 같은 대역을 재면 -51.64 dB [^168] 이고, 저역통과로 대역 안 내용물만 남기면 -134.54 dB [^169] 다. 나딧에서 날개끝 상한은 0.0 Hz [^170] 라 이 대역에 블레이드 도플러가 들어올 자리의 폭이 0 이다.


## 엔진마다 나딧 잔여의 크기가 다르다

같은 자리에서 위상 흔들림 p95 는 우리 팔 1.18° [^171], PathSolver 11.1M 73.73° [^172], 250M 21.68° [^173] 다. 광선을 늘린 팔에서 흔들림이 함께 줄어드는 축이 곧 표본화 축이다.

진폭 쪽도 같은 순서다 — AC/DC 는 우리 팔 -38.31 dB [^174], 11.1M -0.11 dB [^175], 250M -11.35 dB [^176] 다. 도플러가 원리적으로 0 인 자리에서 남는 이 크기는 산란 물리가 아니라 각 엔진의 표본화가 정한다.


## 사각지대는 반각 몇 도짜리 원뿔이다

![ch1_nadir_cone](../outputs/figures/ch1_nadir_cone.png)

**그림 7.** 호버 중인 기체가 나딧에서 몇 도 기울면 프로펠러 변조가 돌아오는가?

호버링하는 기체는 바람과 자세 제어로 늘 몇 도 기운다. 그 몇 도가 아래 표의 어느 줄에 앉느냐가 프로펠러 무늬를 볼지 말지를 정한다.


### 각도 예산 — 원거리장 기하가 주는 표

| 나딧에서 벗어난 각 [°] | 변조 AC/DC [dB] |
|---|---|
| 0.0 | -305.01 |
| 0.5 | -55.85 |
| 1.0 | -44.09 |
| 2.0 | -32.81 |
| 5.0 | -11.88 |
| 10.0 | -1.21 |
| 15.0 | -7.02 |

출처 [^177]

10 m 재현기로 같은 각도를 훑으면 0° 에서 -49.18 dB [^178], 5° 에서 -32.52 dB [^179] 다 — 격자 잡음 바닥 위로 각도 항이 올라오는 자리가 그 사이에 있다.


## 탐지로 옮기면 — 에코는 커지고 블레이드 선만 무너진다

나딧에서 몸체 반사는 -9.32 dBsm [^180] 로 −15° 의 -18.84 dBsm [^181] 보다 크고, 변조 성분은 -47.63 dBsm [^182] 로 −15° 의 -38.47 dBsm [^183] 보다 작다.

10 m 링크버짓에서 전체 SNR 은 나딧 44.39 dB [^149] 대 −15° 34.91 dB [^150] 이고, 블레이드 선 SNR 은 나딧 1.68 dB [^184] 대 −15° 15.24 dB [^185] 다.

머리 위 기체는 **에코로 잡고, 프로펠러 무늬는 기울어진 뒤에 잡는다.** 이 사다리의 절대 σ 규약은 원장이 적어 뒀다 — `σ 절대값은 NOT_VALIDATED(das_fleet_validation) — 상대비교로만 읽어라.`


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 호버 자세 분포를 넣어 나딧 0~5° 를 자세 표본으로 채운다 | 사각지대 원뿔의 입체각이 확정된다 | `benchmark/elevation_sweep_md.py` 앙각 간격 세분 · **새 계산이 필요하다** |
| 나딧 한 점을 조인 격자로 다시 재고 삼분할을 갱신한다 | 격자 잡음 몫 64 % 가 하한에서 실제값으로 좁혀진다 | `benchmark/refute_nadir_mechanism_final.py` R4c · **새 계산이 필요하다** |
| 실측에서 기체를 정면 상공에 띄우고 같은 잣대로 잰다 | 잔여가 수치 인공물인지 실물 신호인지 갈린다 | **절 1** «스윕 규약» → 실측 캠페인 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 46개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^140] | `outputs/verify_nadir_flash.json` | `C_D_geometry.nadir_plane_wave_ac_over_dc_db` | -305 |
| [^141] | `outputs/verify_nadir_flash.json` | `B_decomposition.ours/el-90.ac_over_dc_db` | -38.31 |
| [^142] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.nadir_ac_split.grid_sampling_noise_fraction` | 0.6368 |
| [^143] | `outputs/verify_nadir_flash.json` | `C_D_geometry.offnadir_farfield.-89.5.off_nadir_deg` | 0.5 |
| [^144] | `outputs/verify_nadir_flash.json` | `C_D_geometry.offnadir_farfield.-89.5.ac_over_dc_db` | -55.85 |
| [^145] | `outputs/verify_nadir_flash.json` | `C_D_geometry.offnadir_farfield.-85.0.off_nadir_deg` | 5 |
| [^146] | `outputs/verify_nadir_flash.json` | `C_D_geometry.offnadir_farfield.-85.0.ac_over_dc_db` | -11.88 |
| [^147] | `outputs/refute_nadir_mechanism_final.json` | `R4b_cpu_kernel_replica.range_sweep_ac_over_dc_db.10` | -48.46 |
| [^148] | `outputs/refute_nadir_mechanism_final.json` | `R4b_cpu_kernel_replica.range_sweep_ac_over_dc_db.1000` | -50.31 |
| [^149] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.rows.10m.nadir_snr_total_db` | 44.39 |
| [^150] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.rows.10m.el15_snr_total_db` | 34.91 |
| [^151] | `outputs/elevation_sweep_md.json` | `_meta.range_m` | 10 |
| [^152] | `outputs/elevation_sweep_md.json` | `rows[6].n_poses` | 4096 |
| [^153] | `outputs/elevation_sweep_md.json` | `rows[6].n_missing` | 0 |
| [^154] | `outputs/refute_nadir_mechanism_final.json` | `R2_analytic.geometry_from_mesh.hub_radius_m` | 0.2196 |
| [^155] | `outputs/refute_nadir_mechanism_final.json` | `R2_analytic.per_element.two_way_phase_swing_pp_deg` | 50.55 |
| [^156] | `outputs/refute_nadir_mechanism_final.json` | `R2_analytic.two_blade_cancellation.predicted_ac_over_dc_db` | -40.61 |
| [^157] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.nadir_ac_split.nearfield_fraction` | 0.3112 |
| [^158] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.nadir_ac_split.occlusion_fraction` | 0.052 |
| [^159] | `outputs/refute_nadir_mechanism_final.json` | `R4b_cpu_kernel_replica.corr_ac_sph10_vs_measured` | 0.998 |
| [^160] | `outputs/refute_nadir_mechanism_final.json` | `R4b_cpu_kernel_replica.shift_null.null_p99` | 0.4618 |
| [^161] | `outputs/refute_nadir_mechanism_final.json` | `R4c_grid_ladder.rows.lambda/12.nearfield_only_fit_exponent` | 2.589 |
| [^162] | `outputs/refute_nadir_mechanism_final.json` | `R4c_grid_ladder.rows.lambda/48.nearfield_only_fit_exponent` | 3.74 |
| [^163] | `outputs/refute_nadir_mechanism_final.json` | `R4a_facet_proxy_no_grid_no_occlusion.fit_exponent` | 4.038 |
| [^164] | `outputs/verify_nadir_flash.json` | `A_instrument_audit.hann_mainlobe_halfwidth_hz` | 562.9 |
| [^165] | `outputs/verify_nadir_flash.json` | `A_instrument_audit.band_lo_hz` | 430.1 |
| [^166] | `outputs/verify_nadir_flash.json` | `B_decomposition.ours/el-90.fixed_band_power_db` | -55.68 |
| [^167] | `outputs/verify_nadir_flash.json` | `B_decomposition.ours/el-90.fixed_band_true_over_leakage_db` | 0.01 |
| [^168] | `outputs/refute_nadir_mechanism_final.json` | `R3_window_robustness.ours/el-90.fullrecord_band_share_db` | -51.64 |
| [^169] | `outputs/refute_nadir_mechanism_final.json` | `R3_window_robustness.ours/el-90.fullrecord_band_share_lowpassed_db` | -134.5 |
| [^170] | `outputs/wideband_energy.json` | `cells.ours/el-90.f_tip_hz` | 0 |
| [^171] | `outputs/verify_nadir_flash.json` | `B_decomposition.ours/el-90.phase_dev_p95_deg` | 1.18 |
| [^172] | `outputs/verify_nadir_flash.json` | `B_decomposition.sionna/el-90.phase_dev_p95_deg` | 73.73 |
| [^173] | `outputs/verify_nadir_flash.json` | `B_decomposition.sionna_p250000000/el-90.phase_dev_p95_deg` | 21.68 |
| [^174] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el-90.ac_over_dc_db` | -38.31 |
| [^175] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el-90.ac_over_dc_db` | -0.11 |
| [^176] | `outputs/ch1_elevation_figdata.json` | `cells.sionna_p250000000/el-90.ac_over_dc_db` | -11.35 |
| [^177] | `outputs/verify_nadir_flash.json` | `C_D_geometry.offnadir_farfield` | (7항목 묶음) |
| [^178] | `outputs/refute_nadir_mechanism_final.json` | `R4d_null_width.cpu_replica.-90.0.sph10_ac_over_dc_db` | -49.18 |
| [^179] | `outputs/refute_nadir_mechanism_final.json` | `R4d_null_width.cpu_replica.-85.0.sph10_ac_over_dc_db` | -32.52 |
| [^180] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.sigma_dbsm.nadir_dc` | -9.32 |
| [^181] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.sigma_dbsm.el15_dc` | -18.84 |
| [^182] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.sigma_dbsm.nadir_ac_at_10m` | -47.63 |
| [^183] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.sigma_dbsm.el15_ac` | -38.47 |
| [^184] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.rows.10m.nadir_snr_blade_physical_db` | 1.68 |
| [^185] | `outputs/refute_nadir_mechanism_final.json` | `R5_detection.rows.10m.el15_snr_blade_db` | 15.24 |


---

## 절 5. el 0 에서 광선을 360 배 늘리면 정지 성분은 0.03 dB 안에 모이고, 같은 한 계단이 el −75 의 레벨을 12.55 dB 옮긴다



> ### 한 일
> **el 0 에서 자세당 광선 수를 네 계단 올려 레벨·프로펠러 대역 전력·박자·경로 수가 각각 어디로 가는지 재고, 같은 사다리를 el −30 에서 대조했다.**

### 결과
1. el 0 의 레벨은 광선 11.1 M 에서 -59.65 dB [^186], 4,000 M 에서 -59.68 dB [^187] 로 0.03 dB 폭 안에 모인다 — 광선이 360 배이고 경로는 223 배다.
2. 그 수렴은 el 0 의 성질이다 — 예산을 한 계단(11.1 M → 250 M)만 올려도 el −75 의 레벨은 -127.60 [^188] → -115.05 dB [^189] 로 12.55 dB 움직인다.
3. 같은 네 계단에서 박자는 376.73 [^190] → 50.27 [^191] → 122.12 [^192] → 58.15 Hz [^193] 로 옮겨 다니고, 입력한 f_flash 는 126.67 Hz [^194] 다.
4. el −30 은 앙각 7 점 중 규칙 예산에서 최강선이 어긋나 있던 유일한 자리이고, 예산을 올리면 252.32 [^195] → 126.61 Hz [^196] 로 붙는다.
5. el 0 에서 오른 것은 변조가 아니라 평평한 표본화 바닥이다 — 250 M 판의 프로펠러 대역 몫 -54.61 [^197] 와 대역외 기준띠 -54.63 dB [^198] 가 0.02 dB 차다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 사다리 | 표적·기하·자세 격자·시드를 고정하고 자세당 광선 수만 11.1 M → 250 M → 1,000 M → 4,000 M 으로 올렸다 — 네 계단 모두 자세 4,096 개가 완결(`n_missing` = 0)이다 |
| 레벨 | 자세 4,096 개의 \|E\| 평균을 dB 로 적은 값. 팔마다 정규화가 달라서 **같은 엔진·같은 앙각 안에서만** 견준다 |
| 박자 | 프로펠러 대역 전력의 시간열을 다시 FFT 해 40~400 Hz 안의 최강선을 잡는다 — `benchmark/elevation_sweep_md.py:237` |
| 대조군 | 같은 사다리를 el −30 에서 세 계단(11.1 M · 250 M · 1,000 M) 돌렸다 |
| 시드 | 두 사다리 모두 시드 하나로 돈다(`benchmark/elevation_sweep_md.py:183`). 시드 축의 크기는 40 m 자리의 8 시드 사다리가 따로 잰다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/elevation_sweep_md.py --merge
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/build_ch1_elevation_figs.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_ch1_el0_budget_ladder.py
```

| | |
|---|---|
| 출력 | `outputs/elevation_sweep_md.json`, `outputs/ch1_elevation_figdata.json`, `outputs/figures/ch1_f6_el0_budget_ladder.png` |
| 소요 | 병합·그림 약 1 분 (CPU). 네 계단의 추적 자체는 합계 약 4.3 시간 (GPU 1 장) |
| 비고 | 계단은 `--spp` 로 광선 수를 직접 준 팔이다 — 팔 이름 꼬리의 숫자가 그 값이고, 꼬리가 없는 `sionna` 가 규칙값이다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 3** «상한 위 누설» | 팔을 가르는 잣대가 무엇인가 — f_flash 는 입력값이라 박자는 잣대가 아니다 |
| **절 1** «앙각 스윕 설계» | 병합판 47 행 중 어느 행을 판정에 쓰는가 |

---


## 네 계단은 자세당 광선 수 하나만 흔든다

팔 이름이 곧 예산이다. `sionna` 는 광선 규칙 (R/3)² × 1M 을 그대로 쓴 팔이고 10 m 에서 11,111,111 발 [^199] 이다(`benchmark/elevation_sweep_md.py:89`). 꼬리에 숫자가 붙은 팔(`sionna_p250000000` 등)은 그 수를 직접 준 것이다.

네 계단 모두 자세 4,096 개 [^200] 가 채워져 있다 — 부분 병합 행은 이 절에서 제외한다.


| 팔 | 자세당 경로 수 중앙값 | 레벨 [dB] | 박자 [Hz] | 1 차 − 2 차 [dB] | 추적 시간 [s] |
|---|---|---|---|---|---|
| sionna | 9 | -59.65 | 376.73 | -0.81 | 1,549.9 |
| sionna_p250000000 | 127 | -59.66 | 50.27 | +11.14 | 1,221.3 |
| sionna_p1000000000 | 471 | -59.67 | 122.12 | +15.13 | 4,133.4 |
| sionna_p4000000000 | 2,008 | -59.68 | 58.15 | +7.77 | 8,481.2 |

출처 [^201]

레벨 칸은 0.03 dB 폭 안에 모이고, 그 옆에서 박자 칸은 326 Hz 를 오간다. **수렴한 것과 맞은 것은 다른 일이다.**

네 계단은 시드가 하나씩이라 그 박자 흔들림이 예산 축 하나로 닫히지 않는다 — 예산을 4,000,000,000 발 [^202] 로 고정하고 시드만 8 장 [^203] 바꾼 40 m 자리에서도 최강선의 표준편차가 65.91 Hz [^204] 다.


## el 0 에서 모이는 것은 정지 성분이다

![el0 budget ladder](../outputs/figures/ch1_f6_el0_budget_ladder.png)

**그림 8.** 광선을 360 배 늘리면 el 0 의 레벨과 박자는 각각 어디로 가는가?

왼쪽이 가장 작은 예산 대비 레벨 변화, 가운데가 최강선의 위치, 오른쪽이 자세당 경로 수다. 파랑이 el 0, 빨강이 대조군 el −30 이다.


프로펠러 대역 전력도 레벨과 함께 모인다 — -73.68 [^205] 에서 -73.61 dB [^206] 로 0.07 dB 폭이다. 두 양 모두 **동체 정반사가 지배하는 정지 성분**이고, 그 성분은 광선 몇 발로도 금방 자리를 잡는다.

그 수렴은 el 0 한 점의 성질이다. 예산을 한 계단(11.1 M → 250 M)만 올려도 el −45 의 레벨은 -125.92 [^207] → -124.43 dB [^208], el −75 는 -127.60 [^188] → -115.05 dB [^189] 로 움직인다. 정반사가 꺼지는 앙각에서는 레벨 자체가 예산에 매달린다.


## 오른 것은 변조가 아니라 표본화 바닥이다

el 0 의 변조 대 정지 비는 -83.56 [^209] 에서 -39.65 dB [^210] 로 43.91 dB 올라온다. 올라온 것이 회전 성분인지 잡음 바닥인지는 같은 원장의 세 칸이 가른다.

250 M 판의 프로펠러 대역 몫은 -54.61 [^197] 이고, 블레이드가 원리적으로 못 오는 대역외 기준띠는 -54.63 dB [^198] 로 **0.02 dB 차**다. 그 판의 빗살 선 SNR 최대는 -0.8 dB [^211] 이고 최강선은 50.27 Hz [^191] 로 f_flash 126.67 Hz [^194] 와 어긋난다.

같은 자리 우리 커널은 대역 몫 -15.58 [^212] 대 기준띠 -40.18 dB [^213] 로 24.60 dB 여유가 있고, 빗살 1 차가 41.5 dB [^214] · 최강선이 126.40 Hz [^215] 다. ⇒ el 0 에서 PathSolver 의 AC/DC 가 재는 것은 자세 간 표본화 잡음이고, 우리 커널의 같은 열은 변조 깊이를 잰다.

최강선은 40~400 Hz 안에서 고른다(`benchmark/elevation_sweep_md.py:237`). 11.1 M 계단의 376.73 Hz [^190] 는 f_flash 의 3 배 자리(380.0 Hz)에서 한 칸 안이고, 250 M 계단의 50.27 Hz [^191] 는 빗살 밖의 창 아래 가장자리 값이다.


## el −30 은 같은 사다리에서 박자가 붙는다

| 팔 | 자세당 경로 수 중앙값 | 레벨 [dB] | 박자 [Hz] | 1 차 − 2 차 [dB] | 추적 시간 [s] |
|---|---|---|---|---|---|
| sionna | 6 | -135.45 | 252.32 | -1.44 | 1,944.1 |
| sionna_p250000000 | 211 | -133.82 | 126.61 | +15.53 | 1,376.8 |
| sionna_p1000000000 | 809 | -131.20 | 126.91 | +14.05 | 3,508.9 |

출처 [^201]


11.1 M 계단의 252.32 Hz [^195] 는 f_flash 의 두 배 자리다. 그 계단에서는 1 차 빗살이 2 차보다 -1.44 dB [^216] 라 최강선이 2 차로 넘어간다. 250 M 에서 그 차이가 +15.53 dB [^217] 로 뒤집히면서 최강선이 1 차로 돌아오고, 1,000 M 에서도 126.91 Hz [^218] 로 머문다.

같은 세 계단에서 el −30 의 레벨은 -135.45 [^219] 에서 -131.20 dB [^220] 로 4.25 dB 올라간다 — el 0 에서 모였던 그 양이 여기서는 계속 움직인다.

el −30 은 앙각 7 점 중 규칙 예산에서 최강선이 어긋나 있던 **유일한 자리**다. 나머지 앙각(−15·−45·−60·−75)은 11.1 M 에서 이미 126.45~126.85 Hz 라 예산이 고칠 최강선이 남아 있지 않고, el 0 은 예산을 360 배까지 올려도 최강선이 창 안을 옮겨 다닌다.

⇒ 예산이 최강선을 되돌리는 자리는 변조 몫이 이미 큰 앙각이다 — el −30 의 추적 대역 몫은 -8.48 dB [^221] 이고 el 0 은 -84.94 dB [^222] 다. 두 앙각을 가르는 것은 표적이 아니라 변조 성분의 크기다.


## 박자의 참값은 우리가 넣은 입력이다

f_flash 126.67 Hz [^194] 는 로터 회전수에서 나온 **입력값**이다. 네 로터의 회전수는 3,791.64 [^223] · 3,795.402 [^224] · 3,804.598 [^225] · 3,808.36 rpm [^226] 이고, 2 엽 기준 `rpm / 60 × 2` 로 바꾸면 126.39 ~ 126.95 Hz 로 흩어져 있다 — 참값 한 점은 그 넷의 평균 자리다.

그래서 이 절이 박자에서 읽는 것은 **같은 입력을 되찾는가**까지다. 그 잣대의 성질은 **절 3** «상한 위 누설» 가 앙각 여섯 점에서 다룬다.


## 예산을 고정해도 경로 수는 앙각을 탄다

![ray budget](../outputs/figures/ch1_f5_raybudget.png)

**그림 9.** 같은 광선 예산 아래에서 경로 수는 앙각을 따라 얼마나 달라지는가?

이 스윕은 거리를 10 m 로 고정했으므로 규칙 (R/3)² × 1M 이 주는 예산은 앙각 7 점에 같은 값 하나다(`benchmark/elevation_sweep_md.py:89`). 앙각을 따라 달라지는 것은 예산이 아니라 **찾아낸 경로의 수**다.


규칙값 팔의 경로 수 중앙값은 앙각 7 점에서 6 [^227]~13 개 [^228], 손으로 250 M 을 고정한 팔은 127 [^229]~352 개 [^230] 로 2.8 배 폭이다. 거리와 무관하게 고정한 팔의 폭이 더 넓으므로, 이 앙각 의존은 규칙의 거리 항이 만든 것이 아니다.

250 M 팔에서 가장 적은 칸이 el 0 의 127 개 [^231] 이고 el −30 은 같은 예산에서 211 개 [^232] 다. 같은 비(1.66 배)가 1 G 계단에서도 471 [^233] 대 809 개 [^234] 로 유지되므로, 앙각 의존은 표적을 보는 각도가 만든다.

규칙값 팔이 남기는 문제는 축이 섞이는 것이라기보다 **경로가 너무 적은 것**이다 — 6~13 개뿐이라 250 M 팔이 단조 증가로 그리는 추이가 그 팔에서는 나타나지 않는다. 앙각 축 전체의 경로 수 표는 **절 1** «앙각 스윕 설계» 에 있다.


## 40 m 자리에서는 시드가 같은 크기의 축이다

el 0 사다리는 시드 하나로 돈다. 시드 축의 크기는 40 m·el −15 자리에서 시드 8 장 [^235] 으로 따로 쟀다 — 시드 간 레벨 표준편차가 178,000,000 [^236] 발에서 4.156 dB [^237], 4,000,000,000 [^202] 발에서 1.833 dB [^238] 다.

그 산포는 자세 평균으로 지워지는 종류가 아니다 — i.i.d. 예측 대비 배수가 23.36 [^239] 배와 31.16 [^240] 배다[^241]. 시드는 자세마다 다시 뽑는 잡음이 아니라 **광선 방향 집합 하나**를 고르는 일이다.


같은 판의 박자도 시드마다 갈린다 — 4,000 M 여덟 시드 중 앞의 넷이 127.15 [^242] · 126.21 [^243] · 126.92 [^244] · 252.66 Hz [^245] 이고, 넷째가 2 차 자리로 넘어가 있다.

⚠ 이 시드 판정은 40 m·matrice4e·el −15°·기선 0 한 자리에서 잰 것이다[^246]. 예산 축을 따라 판정 통계가 어떻게 움직이는지는 [리포트 9 절 3 «예산 법칙»](09_microdoppler-limits.ipynb) 이 검출 통계 쪽에서 같은 축으로 다룬다.

그러므로 el 0 의 박자 흔들림에서 예산 몫과 시드 몫을 가르는 일이 다음 단계 표의 첫 줄이다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| el 0 사다리를 시드 8 장으로 다시 돌린다 | 박자 50.27 [^191]~376.73 Hz [^190] 의 흔들림이 시드 성질인지 예산 성질인지 갈린다 | `benchmark/raybudget_seed_ladder.py` 를 10 m·el 0 에 적용 — 새 GPU 계산이 필요하다 |
| 광선 규칙 (R/3)² × 1M 에 앙각 항을 넣어 경로 수를 앙각마다 맞춘다 | 앙각 곡선에서 예산 축이 분리되고, 남는 차이가 표적의 성질로 닫힌다 | `benchmark/elevation_sweep_md.py:89` 의 `rule_spp()` |
| 최강선 탐색창 40~400 Hz 를 f_flash 배수 기준으로 다시 잡는다 | 50.27 [^191] 와 376.73 Hz [^190] 가 창 가장자리에 눌린 값인지가 확정된다 | `benchmark/elevation_sweep_md.py:237` 의 `band_metrics()` |
| el −30 사다리에 4,000 M 계단을 더한다 | 박자가 붙은 뒤에도 레벨이 계속 오르는지, 어느 계단에서 멈추는지가 정해진다 | `benchmark/elevation_sweep_md.py --els -30 --spp 4000000000` — 새 GPU 계산이 필요하다 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 61개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^186] | `outputs/elevation_sweep_md.json` | `rows[7].level_db → sionna/el+0` | -59.65 (파생) |
| [^187] | `outputs/elevation_sweep_md.json` | `rows[35].level_db → sionna_p4000000000/el+0` | -59.68 (파생) |
| [^188] | `outputs/elevation_sweep_md.json` | `rows[12].level_db → sionna/el-75` | -127.6 (파생) |
| [^189] | `outputs/elevation_sweep_md.json` | `rows[26].level_db → sionna_p250000000/el-75` | -115 (파생) |
| [^190] | `outputs/elevation_sweep_md.json` | `rows[7].track.beat_hz → sionna/el+0` | 376.7 (파생) |
| [^191] | `outputs/elevation_sweep_md.json` | `rows[21].track.beat_hz → sionna_p250000000/el+0` | 50.27 (파생) |
| [^192] | `outputs/elevation_sweep_md.json` | `rows[14].track.beat_hz → sionna_p1000000000/el+0` | 122.1 (파생) |
| [^193] | `outputs/elevation_sweep_md.json` | `rows[35].track.beat_hz → sionna_p4000000000/el+0` | 58.15 (파생) |
| [^194] | `outputs/elevation_sweep_md.json` | `_meta.f_flash_hz` | 126.7 |
| [^195] | `outputs/elevation_sweep_md.json` | `rows[9].track.beat_hz → sionna/el-30` | 252.3 (파생) |
| [^196] | `outputs/elevation_sweep_md.json` | `rows[23].track.beat_hz → sionna_p250000000/el-30` | 126.6 (파생) |
| [^197] | `outputs/ch1_elevation_figdata.json` | `cells.sionna_p250000000/el+0.share_track_db` | -54.61 |
| [^198] | `outputs/ch1_elevation_figdata.json` | `cells.sionna_p250000000/el+0.share_track_oob_db` | -54.63 |
| [^199] | `outputs/elevation_sweep_md.json` | `_meta.sionna_spp` | 11111111 |
| [^200] | `outputs/elevation_sweep_md.json` | `rows[7].n_poses → sionna/el+0` | 4096 (파생) |
| [^201] | `outputs/elevation_sweep_md.json` | `rows` | (47행 표) |
| [^202] | `outputs/raybudget_seed_ladder.json` | `cells[1].spp` | 4000000000 |
| [^203] | `outputs/raybudget_seed_ladder.json` | `cells[1].n_seeds` | 8 |
| [^204] | `outputs/raybudget_seed_ladder.json` | `cells[1].sd_beat_hz` | 65.91 |
| [^205] | `outputs/elevation_sweep_md.json` | `rows[7].track.band_power_db → sionna/el+0` | -73.68 (파생) |
| [^206] | `outputs/elevation_sweep_md.json` | `rows[35].track.band_power_db → sionna_p4000000000/el+0` | -73.61 (파생) |
| [^207] | `outputs/elevation_sweep_md.json` | `rows[10].level_db → sionna/el-45` | -125.9 (파생) |
| [^208] | `outputs/elevation_sweep_md.json` | `rows[24].level_db → sionna_p250000000/el-45` | -124.4 (파생) |
| [^209] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el+0.ac_over_dc_db` | -83.56 |
| [^210] | `outputs/ch1_elevation_figdata.json` | `cells.sionna_p250000000/el+0.ac_over_dc_db` | -39.65 |
| [^211] | `outputs/ch1_elevation_figdata.json` | `cells.sionna_p250000000/el+0.comb_snr_db[0]` | -0.8 |
| [^212] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el+0.share_track_db` | -15.58 |
| [^213] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el+0.share_track_oob_db` | -40.18 |
| [^214] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el+0.comb_snr_db[0]` | 41.5 |
| [^215] | `outputs/ch1_elevation_figdata.json` | `cells.ours/el+0.beat_track_hz` | 126.4 |
| [^216] | `outputs/elevation_sweep_md.json` | `rows[9].track.h1_over_h2_db → sionna/el-30` | -1.44 (파생) |
| [^217] | `outputs/elevation_sweep_md.json` | `rows[23].track.h1_over_h2_db → sionna_p250000000/el-30` | 15.53 (파생) |
| [^218] | `outputs/elevation_sweep_md.json` | `rows[16].track.beat_hz → sionna_p1000000000/el-30` | 126.9 (파생) |
| [^219] | `outputs/elevation_sweep_md.json` | `rows[9].level_db → sionna/el-30` | -135.4 (파생) |
| [^220] | `outputs/elevation_sweep_md.json` | `rows[16].level_db → sionna_p1000000000/el-30` | -131.2 (파생) |
| [^221] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el-30.share_track_db` | -8.479 |
| [^222] | `outputs/ch1_elevation_figdata.json` | `cells.sionna/el+0.share_track_db` | -84.94 |
| [^223] | `outputs/elevation_sweep_md.json` | `_meta.rpm_per_rotor[1]` | 3792 |
| [^224] | `outputs/elevation_sweep_md.json` | `_meta.rpm_per_rotor[2]` | 3795 |
| [^225] | `outputs/elevation_sweep_md.json` | `_meta.rpm_per_rotor[3]` | 3805 |
| [^226] | `outputs/elevation_sweep_md.json` | `_meta.rpm_per_rotor[0]` | 3808 |
| [^227] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_npaths_min_max[0]` | 6 |
| [^228] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_npaths_min_max[1]` | 13 |
| [^229] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_p250000000_npaths_min_max[0]` | 127 |
| [^230] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_p250000000_npaths_min_max[1]` | 352 |
| [^231] | `outputs/elevation_sweep_md.json` | `rows[21].npaths_median → sionna_p250000000/el+0` | 127 (파생) |
| [^232] | `outputs/elevation_sweep_md.json` | `rows[23].npaths_median → sionna_p250000000/el-30` | 211 (파생) |
| [^233] | `outputs/elevation_sweep_md.json` | `rows[14].npaths_median → sionna_p1000000000/el+0` | 471 (파생) |
| [^234] | `outputs/elevation_sweep_md.json` | `rows[16].npaths_median → sionna_p1000000000/el-30` | 809 (파생) |
| [^235] | `outputs/raybudget_seed_ladder.json` | `cells[0].n_seeds` | 8 |
| [^236] | `outputs/raybudget_seed_ladder.json` | `cells[0].spp` | 178000000 |
| [^237] | `outputs/raybudget_seed_ladder.json` | `cells[0].sd_level_db` | 4.156 |
| [^238] | `outputs/raybudget_seed_ladder.json` | `cells[1].sd_level_db` | 1.833 |
| [^239] | `outputs/raybudget_seed_ladder.json` | `cells[0].structure.ratio_observed_over_iid` | 23.36 |
| [^240] | `outputs/raybudget_seed_ladder.json` | `cells[1].structure.ratio_observed_over_iid` | 31.16 |
| [^241] | `outputs/raybudget_seed_ladder.json` | `verdict.structure_ko` | ⭐시드는 자세마다 다시 뽑는 잡음이 아니라 **광선 방향 집합 하나**를 고르는 것이다(모든 자세에… |
| [^242] | `outputs/raybudget_seed_ladder.json` | `cells[1].beats_hz[0]` | 127.2 |
| [^243] | `outputs/raybudget_seed_ladder.json` | `cells[1].beats_hz[1]` | 126.2 |
| [^244] | `outputs/raybudget_seed_ladder.json` | `cells[1].beats_hz[2]` | 126.9 |
| [^245] | `outputs/raybudget_seed_ladder.json` | `cells[1].beats_hz[3]` | 252.7 |
| [^246] | `outputs/raybudget_seed_ladder.json` | `verdict.caveat_ko` | ⚠ 이 판정은 40 m·matrice4e·el −15°·기선 0 한 자리에서만 잰 것이다. 다른 거… |
